# Macaque Atlas — Experiment Runner **v3**  (E0, E3–E6)

Wraps the **v19 TEST pipeline unchanged** and runs it over parameter combinations, capturing
**S1–S7 diagnostics**, **per-section + per-fusion-stage timings**, and a ground-truth error.

## What changed from v2

**Eight real defects fixed** (each is labelled `v3 FIX A..H` in the code, so they are greppable):

| | Defect | Consequence |
|---|---|---|
| A | `ef.run_fusion`'s **return value** was discarded | `S7_budget_saturated` / `S7_fit_residual_px` needed polygons for no reason |
| B | `REGISTRY_PATH` frozen at the **first** run's folder | 8001+/9001+ ids allocated monotonically across a batch — **no two runs were comparable** on any region-id metric |
| C | `duplicate_to` only ever **grew** the subject set | `n_subjects=2` with 3 real subjects fused 3; E5's N axis was not an axis |
| D | `drop_arcs` deleted **segments**, not arcs | measured: `frac=0.05` turned 40 arcs into **129 fragments** — E6 never tested orphan policy |
| E | `build_polygons` taken from the sweep | a plan could ask for S7 with polygonize off and silently get nothing |
| F | sweeping `kp_alpha` kept the **stale** `curv_sigma_px` | E4.1 measured alpha against a fixed curvature scale |
| G | `ensure_lines` called **non-existent** propagate functions | the full-stack path propagated nothing; only the cache path had ever worked |
| H | `corrected` read from **state**, and `load_state` falls back across runs | a run that failed to write state silently fused a *previous* run's geometry and reported success |

**New capabilities**

* **Stage independence.** `stages=("parc","pert","fuse","poly")` — any prefix. `parc` gives `P_*`, `pert` adds `X_*`, `fuse` adds `S1–S4`, `poly` adds `S5, G_*, S6, S7`. Note `fuse` and `poly` are **one call** to `ef.run_fusion` (polygonize consumes the fused arcs), so `poly` means fuse+polygonize.
* **`X_tdev` — deviation from the unperturbed template.** In `mode="anti"` the signs sum to zero, so a correct fusion **must** come back to the template. v2 never measured against that known ground truth; every E3/E4 number was a self-consistency number. `X_tdev` needs no polygons, which is what makes a cheap fuse-only screening pass scientifically meaningful.
* **Seed replication** with `-seedX` prefixes, triggered by `mode="indep"` **or** `drop_frac > 0` **or** `repeat_seeds=True`.
* **Append-only `runs.jsonl`** as the single source of truth, doubling as the resume registry.
* **`strict_topo` is forced `True`** on every run.

**You edit two cells:** §3 (the registry) and its `ACTIVE` list.


## 1 · TEST v19 pipeline cells (verbatim)

Copied **unchanged** from `Macaque_MRI_Atlas_TEST_lightweight_v19.ipynb`. The only edits are mechanical: each cell's bottom-of-cell *driver call* is commented out (`[runner-controlled]`) because the runner decides when to call it, plus the two marked changes in CELL A (`WORK_TEST` -> v4, `strict_topo` -> True).

In [ ]:
#@ IMPORTABLE
# =====================================================================================
# CELL A  --  LIGHTWEIGHT TEST SETUP   (reuse CELL 2 outputs from Drive; NO rebuild)
# =====================================================================================
# This is the TESTING notebook. It starts from CELL 3 (scans already registered to the
# template in a prior full run) and only touches ONE slice per subject so the whole
# propagate -> export -> average path runs in seconds.
#
# What this cell does (and does NOT do):
#   * MOUNTS Drive and reuses the SAME template + registration outputs your full pipeline
#     already wrote. It does NOT rebuild the DB09 atlas (CELL 0b / CELL 2) and it does NOT
#     touch or delete any cache under work/  (read-only use of that directory's products).
#   * Reads the cached objects CELL 2 stored: TEMPLATE_LABELS, TEMPLATE_LUT, TEMPLATE_T1,
#     and combined_lut. These are recovered from Drive by globbing the template_db09 folder,
#     so you do not have to re-run CELL 0a/0b in this session.
#   * Creates a SEPARATE working directory  macaque_atlas/work_test/  for everything this
#     test notebook writes (state, propagated txt, SVG export, fused output). Your real
#     work/ directory is never written to.
from google.colab import drive; drive.mount('/content/drive')
!pip -q install antspyx nibabel numpy scipy shapely matplotlib scikit-image svgpathtools

import os, glob, json, time, pickle, subprocess, sys, csv, colorsys
import numpy as np
import nibabel as nib
import ants
from pathlib import Path

# ---- paths (same Drive layout as the full pipeline) ---------------------------------
DRIVE_ROOT  = "/content/drive/My Drive/macaque_atlas"
WORK        = f"{DRIVE_ROOT}/work_TESTv5"          # READ-ONLY here: source of registration cache
WORK_TEST   = f"{DRIVE_ROOT}/work_TESTv5"
                                            # v5: fresh parent for METRIC_KEY.csv + _results/
SLICE_AXIS  = 1                             # coronal = axis 1 (unchanged from full pipeline)
TEMPLATE    = "DB09"

# ---- run directory ------------------------------------------------------------------
import re

RUN_PREFIX = "test_v20_full2"          # <-- CHANGE ME EACH RUN (this is now a FOLDER NAME)

def _safe_dirname(s: str) -> str:
    """A run name has to survive being a directory on Drive. '/' and whitespace would
    silently create nested folders or break globbing, so anything outside a conservative
    filename set becomes '_'. Empty -> 'default' (there is no un-foldered mode any more:
    a bare WORK_TEST/atlas/ would let two runs overwrite each other, which is the exact
    problem the prefix existed to prevent)."""
    return re.sub(r"[^A-Za-z0-9._+-]", "_", s) if s else "default"

RUN_DIR = f"{WORK_TEST}/{_safe_dirname(RUN_PREFIX)}"   # <<< everything this run writes

def _pfx(name: str) -> str:
    """KEPT AS A NO-OP so the ~15 existing `_pfx(...)` call sites do not all have to change
    (and so any older code you paste in still runs). The run is separated by RUN_DIR now,
    not by the filename. Delete the call sites at your leisure."""
    return name

MODULES_DIR = f"{DRIVE_ROOT}/modules"       # boundary_graph.py + edge_fusion.py live here
if MODULES_DIR not in sys.path:
    sys.path.insert(0, MODULES_DIR)
TPL_DIR = f"{DRIVE_ROOT}/template"

def _find_one(pattern, what="file"):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matched ({what}): {pattern}")
    return hits[0]

# ---- reference volumes: the WORKING GRID written by REGISTRATION_ONLY CELL 3 ---------
SUBJ_DIR        = f"{WORK}/reg_subjgrid"
TEMPLATE_LABELS = _find_one(f"{SUBJ_DIR}/labels_in_subjgrid.nii.gz", "subject-grid labels")
TEMPLATE_LUT    = TEMPLATE_LABELS.replace(".nii.gz", "_LUT.csv")
TEMPLATE_T1     = _find_one(f"{SUBJ_DIR}/mri_in_subjgrid.nii.gz", "subject-grid template MRI")
SUBJ_GRID       = json.load(open(f"{SUBJ_DIR}/grid.json"))
assert os.path.exists(TEMPLATE_LUT), f"LUT missing next to labels: {TEMPLATE_LUT}"

print(f"[grid] {SUBJ_GRID['target_shape']} @ {SUBJ_GRID['target_zooms']} mm; "   # TROUBLESHOOTING PRINT
      f"template slice -> working slice = round(idx x {SUBJ_GRID['slice_index_scale']:.4f})")

# rebuild combined_lut (id -> abbrev/name/rgb_hex) from the LUT CSV CELL 2 wrote,
# so downstream code that references combined_lut works without re-running CELL 2.
combined_lut = {}
with open(TEMPLATE_LUT, newline="") as f:
    for row in csv.DictReader(f):
        try: rid = int(float(row["id"]))
        except (KeyError, TypeError, ValueError): continue
        combined_lut[rid] = {"abbrev": row.get("abbreviation", str(rid)),
                             "name":   row.get("name", str(rid)),
                             "rgb_hex": row.get("color_hex", ""),
                             "kind":   row.get("type", "region")}

# ---- helper modules -----------------------------------------------------------------
# edge_fusion.py is THE fusion path (arc/node topology + SATM key points).
# contour_fusion.py is now LEGACY -- kept ONLY as the A/B baseline and for the synthetic
# circle sanity check in CELL 9. Do not call fuse_slice_contours() on real slices.
for _m in ("boundary_graph.py", "edge_fusion.py", "contour_fusion.py"):
    if not os.path.exists(f"{MODULES_DIR}/{_m}"):
        raise FileNotFoundError(f"{_m} not found in {MODULES_DIR}. Upload it there before this cell.")
import boundary_graph as bg
import contour_fusion as cf
import edge_fusion as ef

# =====================================================================================
# FUSION PARAMETERS   (AFAM Part II.  Units are VOXELS; 1 voxel == 1 SVG px here.)
# =====================================================================================
# TUNE IN THIS ORDER -- it is not arbitrary, and the first knob can make things WORSE.
#
#   1. kp_alpha        sets the SCALE OF PRESERVED DETAIL.
#   2. fit_tol_px      sets rendering fidelity, independently.
#   3. node_match_max  from YOUR measurement (~3 x S3_node_disp_p95_px, printed each run).
#   4. curv_lambda / kp_gamma  only if matching misfires.
#
# MEASURED (synthetic notch, 6px wide x 20px deep, displaced 16px between tracers):
#
#   config                              notch   peri_dev   avg_gl   round_dev
#   keypoints OFF (= Karcher mean)      20.0px   -0.0425   -0.2321    0.1015
#   kp_alpha = 8  (COARSER than notch)  10.0px   -0.0562   -0.2258    0.1831   <-- WORSE than OFF
#   kp_alpha = 4  (finer than notch)    20.0px   -0.0234   -0.1428    0.0568
#   kp_alpha = 2                        20.0px   -0.0158   -0.1053    0.0463
#
# A kp_alpha COARSER than the feature is worse than having no key points at all: the greedy
# alpha-separation rule then keeps only ONE of the notch's two tip corners, anchors it
# asymmetrically, and pulls the notch apart. MEASURE YOUR SMALLEST NOTCH AND SET ALPHA BELOW IT.
FUSION_PARAMS = ef.FusionParams(
    # --- 1. anchors (F1 / SATM Step 1+2) --------------------------------------------
    keypoints      = False,
    kp_alpha       = 4.0,     # px. Min spacing between key points. TUNE THIS FIRST.
    kp_curv_thresh = 0.05,    # 1/px. Both signs kept: -ve = concavity (SATM's CURV; Fig 1b
                              #       shows concavities are smoothed away without it, and an
                              #       atlas is nothing but notches, clefts and sulci).
    kp_dmax        = 12.0,     # px. Beyond this, key points are REJECTED, not force-matched.
    kp_gamma       = 1/3,     # paper's best (SATM MM3)
    kp_l_scale     = None,    # None -> mean arc length, so gamma*l is in PX and actually does
                              # something. Set 1.0 for AFAM's literal (fractional) formula.

    # --- 2. sampling (F1a) -- REPLACES N_FUSION_SAMPLES / N_BOUNDARY_ANCHORS ----------
    fit_tol_px  = 0.05,       # max chord deviation. DRIVES the point count (sagitta bound).
    n_min_seg   = 2,          # a straight segment needs exactly 2 points
    n_max_seg   = 200,        # clamp. Saturation is DIAGNOSTIC -> S7_budget_saturated.
    curv_lambda = 0.5,        # 0 = arc-length param, 1 = pure turning param

    # curvature SCALE. None -> min(1.0, kp_alpha/4). Do not raise above kp_alpha/4: the
    # curvature scale must be FINER than the detail scale you claim to preserve.
    curv_sigma_px  = None,
    curv_sample_px = 0.25,

    # --- 3. correspondence (F2, F3) ---------------------------------------------------
    node_tol       = 0.05,     # WITHIN a subject: endpoints this close are one junction
    node_match_max = 35.0,     # ACROSS subjects: refuse to pair junctions further than this
    strict_topo    = True,     # PROJECT RULE: ALWAYS True. Region-adjacency divergence is a
                              # HARD STOP, not a warning. E2.6 cannot abort without it, and
                              # with it every E3/E5 run reports its own abort as a result.
    strict_topo_min = 0.98,

    # --- 4. policy (F4) ---------------------------------------------------------------
    orphan_policy   = "passthrough",  # passthrough | drop | reference
    min_arc_support = 1,
    # PER-SLICE reference subject. TEST-NOTEBOOK ONLY toggle (this notebook fuses ONE slice, so a
    # single value is meaningful here; the full pipeline would need per-slice logic).
    #   set to a subject id (e.g. "NHP1") to FORCE that subject as the reference this slice;
    #   set to "AUTO" (or None) to let the auto ladder decide AFTER phantom injection:
    #     most nodes  ->  fewest phantoms injected  ->  first subject.
    reference_sid   = "AUTO",

    # --- representation (F1b) ---------------------------------------------------------
    representation = "polyline",   # "spline" also works (per-segment fit -> corners survive).
    spline_smooth  = 0.0,          # Measure both: flip and compare S7_peri_dev / S7_round_dev.

    # --- closed loops (islands) -------------------------------------------------------
    loop_align = "fft",       # "fft" (exact global optimum) | "coarse" (stepped search)

    # --- rebuild (F5): THE POLYGONS ARE THE ATLAS -------------------------------------
    outer_code         = 0,      # DB09 background
    unlabeled_id_start = 8001,   # a face the code-set rule CANNOT decide becomes an explicit
                                 # UNLABELED region from here up, so a QA gap appears ON the
                                 # atlas instead of vanishing into the unbounded face.
    new_id_start       = 9001,   # a region the EXPERT drew (closed path, no parseable id)
    node_arcs      = True,       # unary_union before polygonize (crossing arcs MUST be noded)
    repair_dangling  = True,
    dangle_repair_px = 1.5,
    min_face_area    = 0.0,

    # --- ADD: stage 3-5 gate + informed-metrics toggle -----------------------------------
    build_polygons        = False,   # <<< FALSE = STOP AT THE FUSED LINES (no polygons yet).
                                     #     Flip True when you want the atlas polygons back.
    diag_ignore_unlabeled = True,    # S6/S7 skip the 8001+ QA-gap faces so the shape metrics
                                     #     describe the regions that actually fused. Harmless
                                     #     when build_polygons=False (no S6/S7 runs anyway).
    # -------------------------------------------------------------------------------------

    # --- precision (F6 / SATM C8): NOTHING is quantised internally ---------------------
    flatten_px  = 0.05,       # Bezier flattening on IMPORT. MUST be <= fit_tol_px.
    grid_write  = None,       # quantisation ON WRITE ONLY. None = FULL FLOAT.
                              # DEVIATION FROM AFAM: AFAM says GRID = 0.1. That predates
                              # FIT_TOL_PX = 0.05 -- a 0.1px write grid has a max error of
                              # 0.05px, i.e. it throws away exactly the tolerance the sagitta
                              # bound was just solved for, and write quantisation becomes the
                              # DOMINANT error term. Set 0.1 if you want AFAM literally.
    svg_decimals = 4,

    # --- metrics: COMPARATIVE DIAGNOSTICS (S6 + S7) -----------------------------------
    # These compare the FUSED shape against each SUBJECT's own shape, so they force a rebuild of
    # every subject's atlas (the polygonize step, once PER SUBJECT) plus per-region shapely work.
    # That subject-rebuild is the single biggest slice of the runtime. The atlas itself and the
    # cheap pipeline-correctness checks (S1..S5) DO NOT need any of it.
    #
    #   run_comparative_diag = False  -> FAST QA-geometry pass. One rebuild instead of
    #                                    (1 + n_subjects); S6/S7 are simply not computed. Use this
    #                                    while you are iterating on the LINES and just want to see
    #                                    the fused atlas quickly.
    #   run_comparative_diag = True   -> compute S6/S7. Then the three EXPENSIVE per-region metrics
    #                                    below are each independently switchable. The other S7
    #                                    numbers (peri_dev M2, round_dev M1, topo_delta M6) are
    #                                    ~free once the shared unions exist, so they always run.
    #
    # MEASURED (120-region test slice, 2 subjects, on this machine):
    #   run_comparative_diag = False .......................... 3.0 s
    #   True, all three metrics ............................... 6.0 s
    #   True, avg_gl + curv_ks (skele off; the default) ....... 5.9 s
    #   True, curv_ks only .................................... 4.7 s
    #   True, no expensive metrics ............................ 4.4 s
    run_comparative_diag = True,   # <<< MASTER SWITCH. False = fast QA geometry, no S6/S7.
    metrics_avg_gl  = True,   # M3' protrusion preservation (convex-hull distance).  ~1.0 s / slice
    metrics_curv_ks = True,   # M7 curvature-distribution KS test.                     ~0.6 s / slice
    metrics_skele   = False,  # M4 skeleton/hull ratio. Needs skimage + rasterisation. ~0.2 s / slice
)

REGISTRY_PATH    = f"{RUN_DIR}/atlas/region_registry.json"
FUSION_RUN_ID    = 1          # bump each time you re-run fusion on new QA output

# ---- per-run scratch tree (real work/ is never written) -----------------------------
# Each subfolder is created by whatever writes into it (save_state, CELL 5, CELL 6,
# _graph_to_svg, ef.save_registry), so a run folder only contains the folders it used.

def save_state(name, obj):
    os.makedirs(f"{RUN_DIR}/state", exist_ok=True)          # <<< EDIT: create-on-write
    with open(f"{RUN_DIR}/state/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)

def load_state(name, default=KeyError):
    """Lookup ladder, first hit wins:
       1. THIS RUN's state (RUN_DIR/state/) -- the normal path.
       2. LEGACY prefixed state still sitting flat in work_test/state/ -- so runs you did
          before this reorganisation are still readable without re-running them.
       3. LEGACY un-prefixed state in work_test/state/.
       4. The REAL shared work/ cache: 'regs' and 'preproc' live there and are never
          per-run, which is why that entry must stay LAST but must stay."""
    cands = [f"{RUN_DIR}/state/{name}.pkl",
             f"{WORK_TEST}/state/{RUN_PREFIX}_{name}.pkl",
             f"{WORK_TEST}/state/{name}.pkl",
             f"{WORK}/state/{name}.pkl"]
    path = next((p for p in cands if os.path.exists(p)), None)
    if path is None:
        if default is KeyError:
            raise FileNotFoundError(f"state '{name}' not found (tried: {cands})")
        return default
    print(f"[load_state] '{name}' <- {path}")   # TROUBLESHOOTING PRINT
    with open(path, "rb") as f:
        return pickle.load(f)

def report_outputs(cell_name, files=(), state_keys=(), checks=()):
    print(f"\n=== {cell_name}: outputs ===")
    for f in files:
        ok = os.path.exists(f); sz = os.path.getsize(f) if ok else 0
        print(f"  {'OK ' if ok and sz>0 else '!! '}{f}  ({sz} bytes)")
    for k in state_keys:
        p = f"{RUN_DIR}/state/{k}.pkl"
        print(f"  {'OK ' if os.path.exists(p) else '!! '}state/{k}.pkl")
    for label, okc, hint in checks:
        print(f"  {'PASS' if okc else 'WARN'}: {label}" + ("" if okc else f"  -> {hint}"))
    print("=" * (len(cell_name) + 16))

print("Lightweight test setup done.")
print("  template labels :", TEMPLATE_LABELS)
print("  template LUT    :", TEMPLATE_LUT, f"({len(combined_lut)} regions)")
print("  template MRI    :", TEMPLATE_T1)
print("  reading reg cache from :", f"{WORK}/reg")
print("  work_test parent :", WORK_TEST)
print("  THIS RUN writes  :", RUN_DIR)
print(f"  RUN_PREFIX = '{RUN_PREFIX}'  -> one folder per run; filenames are now BARE, e.g.")
print(f"      {RUN_DIR}/state/propagated.pkl")
print(f"      {RUN_DIR}/atlas/fused_slice_042.svg")
print(f"\n  fusion: kp_alpha={FUSION_PARAMS.kp_alpha}  fit_tol_px={FUSION_PARAMS.fit_tol_px}  "
      f"curv_sigma_px={FUSION_PARAMS.curv_sigma_px}  rep={FUSION_PARAMS.representation}")
print(f"          orphan={FUSION_PARAMS.orphan_policy}  strict_topo={FUSION_PARAMS.strict_topo}  "
      f"unlabeled_id_start={FUSION_PARAMS.unlabeled_id_start}")
if FUSION_PARAMS.run_comparative_diag:
    _mx = [m for m, on in (("avg_gl", FUSION_PARAMS.metrics_avg_gl),
                           ("curv_ks", FUSION_PARAMS.metrics_curv_ks),
                           ("skele", FUSION_PARAMS.metrics_skele)) if on]
    print(f"          diagnostics: S1-S5 + S6/S7 ON  (expensive metrics: {', '.join(_mx) or 'none'})")
else:
    print(f"          diagnostics: S1-S5 only  (S6/S7 OFF -> fast QA geometry; "
          f"set run_comparative_diag=True for shape metrics)")


In [ ]:
#@ IMPORTABLE
# =====================================================================================
# CELL 3  --  IMPORT PRE-SKULL-STRIPPED SCANS   (unchanged params; N-agnostic)
# =====================================================================================
# Same as the full pipeline's CELL 1: assemble already-preprocessed brains into `preproc`.
# Add or remove entries in SUBJECT_SCANS freely -- everything downstream loops over whatever
# subjects are present, so this test runs with 2 subjects now and any number later.
import os
import nibabel as nib

# --- explicit scan paths (already skull-stripped) ------------------------------------
SUBJECT_SCANS = {
    "NHP1": f"{DRIVE_ROOT}/scans/nifti_out/NHP1/NHP1_scan9_reco1_zfix_skullstripped_cropped_final.nii.gz",
    "NHP2": f"{DRIVE_ROOT}/scans/nifti_out/NHP2/NHP2_scan3_reco1_skullstripped_cropped_final.nii.gz",
    # add/remove subjects here; keep this dict identical to TEST CELL 3.
}

def import_preprocessed():
    pp = {}
    for sid, path in SUBJECT_SCANS.items():
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            raise FileNotFoundError(f"{sid}: scan not found or empty: {path}")
        nib.load(path)   # header-only sanity load
        pp[sid] = {"brain": path, "mask": None, "pre": path, "warpedtpl": None}
    save_state("preproc", pp)
    report_outputs("CELL 3 import",
                   files=[d["brain"] for d in pp.values()],
                   state_keys=["preproc"])
    print(f"\nImported {len(pp)} pre-skull-stripped subjects:")
    for sid, d in pp.items():
        print(f"  {sid}: {d['brain']}")
    return pp

#import_preprocessed()

In [ ]:
#@ IMPORTABLE
# =====================================================================================
# CELL 4  --  LOAD AFFINE REGISTRATION FROM CACHE   (no re-registration in test mode)
# =====================================================================================
# You already registered both scans to the template in a full run, so this test does NOT
# call ants.registration. It only VALIDATES that the cached warped image + transform files
# CELL 2 persisted into work/reg/ still exist, then loads the 'regs' state so CELL 5 can use
# the same warped underlays and transforms. work/ is read only here.
import os, pickle
import ants, nibabel as nib, numpy as np

def registration_metrics(warped_path, template_path=None):
    template_path = template_path or TEMPLATE_T1
    f = ants.image_read(template_path); m = ants.image_read(warped_path)
    fa, ma = f.numpy().ravel(), m.numpy().ravel()
    a = (fa - fa.mean())/(fa.std()+1e-9); b = (ma - ma.mean())/(ma.std()+1e-9)
    ncc = float(np.mean(a*b))
    fg = (fa > 0) | (ma > 0)
    hist,_,_ = np.histogram2d(fa[fg], ma[fg], bins=64)
    pxy = hist/(hist.sum()+1e-9); px = pxy.sum(1); py = pxy.sum(0)
    Hx = -np.sum(px[px>0]*np.log(px[px>0])); Hy = -np.sum(py[py>0]*np.log(py[py>0]))
    Hxy = -np.sum(pxy[pxy>0]*np.log(pxy[pxy>0]))
    return {"ncc": ncc, "nmi": float((Hx+Hy)/(Hxy+1e-9))}

def cell4_load_registration():
    pp = load_state("preproc")
    regs = {}
    missing = []
    for sid in pp:
        warped = f"{SUBJ_DIR}/{sid}_in_subjgrid.nii.gz"
        tfm    = f"{WORK}/reg/{sid}_transforms.pkl"
        if not (os.path.exists(warped) and os.path.exists(tfm)):
            missing.append(sid); continue
        with open(tfm, "rb") as f: saved = pickle.load(f)
        if not all(os.path.exists(p) for p in saved["fwd"] + saved["inv"]):
            missing.append(sid); continue
        regs[sid] = {**saved, "warped": warped, "seconds": 0.0, "cached": True}
    if missing:
        raise FileNotFoundError(
            f"No resampled registration for {missing} under {SUBJ_DIR}. "
            f"Run the REGISTRATION_ONLY notebook's CELL 2 then CELL 3 for these subjects.")
    save_state("regs", regs)
    checks = []
    lab_shape = nib.load(TEMPLATE_LABELS).shape
    for sid, r in regs.items():
        ncc = registration_metrics(r["warped"])["ncc"]
        checks.append((f"{sid} template overlap (NCC={ncc:.2f})", ncc > 0.5,
                       "low overlap -> re-check the full run's CELL 2 for this subject"))
        checks.append((f"{sid} on the label grid {lab_shape}",
                       nib.load(r["warped"]).shape == lab_shape,
                       "grid mismatch -> re-run the registration notebook's CELL 3"))
    report_outputs("CELL 4 load registration (cache)",
                   files=[r["warped"] for r in regs.values()],
                   state_keys=["regs"], checks=checks)
    return regs

#cell4_load_registration()

In [ ]:
#@ IMPORTABLE
# =====================================================================================
# CELL 5  --  PROPAGATE LABELS  ->  each subject, for ONE toggled slice only
# =====================================================================================
# Same tracing/graph machinery as the full pipeline's CELL 3, but restricted to a SINGLE
# slice so the whole path runs in seconds. Because the pipeline is TEMPLATE-SPACE (every
# subject shares the template grid after CELL 4), the template graph for the chosen slice is
# built ONCE and handed to each subject as an identical copy. That is exactly why the SAME
# slice with the SAME labels is propagated to BOTH subjects -- which is what makes the
# averaging test meaningful (identical inputs must fuse back to themselves).
#
# ----------------------------- TOGGLES / PARAMS --------------------------------------
# -- slice selection ------------------------------------------------------------------
SLICE_INDEX  = 50           # <<< THE TOGGLE. Which coronal slice to test on. Subject grid:
                            # the old template index 249 is index 50 at 0.75 mm slices.
SLICE_RANGE  = (12, 78)    # advisory guard-rail: SLICE_INDEX should sit in this range
#
# -- boundary smoothing --------------------------------------------------------------
# WHY THIS CHANGED (v11 -> v12): tracing a label image gives an axis-aligned PIXEL STAIRCASE
# (only 90-degree corners). The old scheme ran Chaikin corner-cutting (SMOOTH_ITERS) and then
# RE-SIMPLIFIED the result at simplify_tol*0.5. That second step UNDOES the smoothing: Douglas-
# Peucker keeps the sharpest points and drops the ones on straights, and on a rounded staircase
# the sharpest points ARE the original step corners -- so the staircase came back. Measured on a
# known circle: Chaikin(5) alone -> 0 sharp corners (good) but 10k points; Chaikin(5)+re-simplify
# -> 219 sharp corners (the jaggies you saw). Paying for Chaikin, then throwing it away.
#
# The fix is a low-pass filter on the traced coordinates (Gaussian), THEN one simplify. That
# produces an actual smooth contour at a sane point count (measured: 33 points, 0 sharp corners,
# and the LOWEST deviation from the true circle of anything tried). This is the SAME idea as the
# old EXPORT SMOOTH_SIGMA that was removed -- but that one ran at EXPORT (so you'd QA a curve that
# was not the data) and used sigma=3.0 (larger than kp_alpha, so it erased real notches). Here it
# runs at TRACE time (the graph, the QA, and the fusion all see the same smooth curve) and sigma
# is small (< kp_alpha), so notches survive. Set SMOOTH_SIGMA=0.0 for the exact pixel staircase.
SMOOTH_BOUNDARIES = True   # False = exact pixel staircase; True = Gaussian-smoothed contour
SMOOTH_SIGMA      = 2.0    # px. Low-pass scale on the traced boundary. Keep BELOW kp_alpha (=4)
                           # so the smoothing cannot erase a notch the fusion is meant to preserve.
                           # 0.75-1.5 rounds the staircase; do not exceed ~kp_alpha/3.
#
# -- anchor points (TRACE stage)   *** N_BOUNDARY_ANCHORS IS GONE ***  [BEYOND AFAM] ------
# The old N_BOUNDARY_ANCHORS = 24 resampled every boundary to a FIXED count, EVENLY SPACED BY
# ARC LENGTH. That is the same mistake AFAM F1a names at the fusion stage, and it is worse here
# because it runs BEFORE the expert QA: whatever it destroyed was gone before fusion ever saw it.
# Evenly-spaced-by-arc-length points do not land on corners, so a right-angle notch got its
# vertices MOVED OFF the corner and rounded away.
#
# The fix is NOT to swap in F1a's sagitta budget here. `geom.simplify(simplify_tol)` is
# Douglas-Peucker, which ALREADY allocates vertices by curvature -- optimally, and by deviation
# rather than by count. Running an arc-length resample on top of DP only undoes it. (F1a belongs
# at the FUSION stage, where AFAM puts it: it replaces N_FUSION_SAMPLES, not this.)
#
# So the traced+simplified vertices ARE the geometry. simplify_tol is the fidelity knob.
# TRACE_MAX_HANDLES only exists for Illustrator ERGONOMICS -- 400 handles on one boundary is
# unusable to drag. When a boundary exceeds it, the tolerance is RAISED (more DP), which keeps
# the corners and drops points along the straights. It never redistributes by arc length. The
# resulting deviation is measured and reported, so the cost of the ergonomics is visible.
TRACE_MAX_HANDLES = None   # None = keep the DP vertices (recommended: lossless within
                           # simplify_tol). An int caps handles per boundary, LOSSILY.
#
# -- detail of labels propagated -------------------------------------------------------
#   "none"   -> boundaries only; no region text is emitted at export
#   "abbrev" -> region NUMBER + short abbreviation (e.g. "Cd")           [current behavior]
#   "full"   -> abbreviation + full name (e.g. "Cd - Caudate nucleus")
LABEL_DETAIL = "abbrev"
#
# -- single vs double boundary lines ---------------------------------------------------
# "single" (current): one shared line sits on the border between region A and region B.
# "double": the border is drawn as TWO parallel lines, one offset to each side, each colored
#           for its own region and each HALF the stroke width, so every region ends up fully
#           enclosed by a continuous boundary in its OWN color. This is a RENDER/EXPORT choice;
#           geometry/topology is unchanged, and CELL 7's importer COLLAPSES the pair back to
#           one line on the way in (see _collapse_double_lines).
LINE_MODE = "single"       # "single" | "double"
LINE_DOUBLE_OFFSET = 0.35  # voxel offset of each half-line from the true border (double only)
#
# -- hemisphere split -------------------------------------------------------------------
#   "whole" -> both hemispheres (current behavior; no mirror-back needed)
#   "left"  -> keep L-R rows below the midline
#   "right" -> keep L-R rows at/above the midline
HEMISPHERE = "left"       # "whole" | "left" | "right"
# -------------------------------------------------------------------------------------


def _hemisphere_mask(label_slice, hemisphere):
    """Zero out the half we are NOT keeping. axis 0 of the raw slice = L-R; cut at its
    midline. Returns (masked_slice, midline_lr) where midline_lr is the L-R row index used
    as the mirror axis in CELL 8. hemisphere='whole' returns the slice unchanged, mid=None.

    NOTE: zeroing the far half means the tracer sees label 0 there, so it emits (rid, 0)
    boundaries ALONG THE MIDLINE. That is what keeps every kept-hemisphere region CLOSED,
    which is what polygonize needs. Do not 'optimise' that away."""
    if hemisphere == "whole":
        return label_slice, None
    H_lr = label_slice.shape[0]
    mid = H_lr / 2.0
    out = label_slice.copy()
    rows = np.arange(H_lr)
    if hemisphere == "left":
        out[rows >= mid, :] = 0          # keep rows below midline
    elif hemisphere == "right":
        out[rows < mid, :] = 0           # keep rows at/above midline
    else:
        raise ValueError(f"HEMISPHERE must be 'whole'|'left'|'right', got {hemisphere!r}")
    return out, mid

if not (SLICE_RANGE[0] <= SLICE_INDEX <= SLICE_RANGE[1]):
    print(f"  !! note: SLICE_INDEX={SLICE_INDEX} is outside advisory SLICE_RANGE {SLICE_RANGE}; "
          f"running anyway.")


def view_template_slice(slice_index=None, save=None):
    """DEBUG HELPER: render one slice of the label volume so you can confirm SLICE_AXIS and
    see how many regions appear. Pass the slice you intend to test."""
    import matplotlib.pyplot as plt
    lab = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    if slice_index is None:
        slice_index = SLICE_INDEX
    sl = np.take(lab, slice_index, axis=SLICE_AXIS)
    print(f"slice {slice_index}: shape {sl.shape}, "
          f"{len(np.unique(sl))-1} regions present (ids {sorted(np.unique(sl))[:10]}...)")
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(np.rot90(sl), cmap="nipy_spectral"); ax.set_title(f"labels, slice {slice_index}")
    ax.axis("off")
    if save: fig.savefig(save, dpi=150, bbox_inches="tight")
    return slice_index


def _smooth_ring(coords, sigma):
    """Low-pass (Gaussian) smooth of a traced boundary polyline, to turn the axis-aligned pixel
    staircase into a real contour BEFORE it is simplified. Replaces the old Chaikin+re-simplify,
    which cancelled itself out (see the note at the top of this cell).

    - CLOSED rings: filtered with wrap-around, so the seam is smooth and no corner is special.
    - OPEN arcs (between triple-point junctions): the two ENDPOINTS are held fixed (mode='nearest'
      then endpoints restored), so shared junctions stay welded across the regions that meet there.
    sigma is in POLYLINE-VERTEX units; the traced staircase has ~1 vertex/px, so sigma~=px here."""
    from scipy.ndimage import gaussian_filter1d
    pts = np.asarray(coords, float)
    if sigma <= 0 or len(pts) < 5:
        return pts
    closed = np.allclose(pts[0], pts[-1])
    if closed:
        ring = pts[:-1]
        x = gaussian_filter1d(ring[:, 0], sigma, mode="wrap")
        y = gaussian_filter1d(ring[:, 1], sigma, mode="wrap")
        out = np.column_stack([x, y])
        return np.vstack([out, out[0]])
    x = gaussian_filter1d(pts[:, 0], sigma, mode="nearest")
    y = gaussian_filter1d(pts[:, 1], sigma, mode="nearest")
    out = np.column_stack([x, y])
    out[0], out[-1] = pts[0], pts[-1]          # keep junction endpoints exactly put
    return out


def _simplify_to_budget(coords, tol_px, max_pts=None):
    """Douglas-Peucker to at most max_pts vertices, KEEPING THE CORNERS.  [BEYOND AFAM]

    DP drops the vertex whose removal costs the least deviation, so straights collapse and
    corners survive -- which is exactly "allocate vertices by curvature", done optimally and by
    DEVIATION rather than by count. Contrast the old fixed-count arc-length resample, which
    spaced points evenly and therefore landed them BETWEEN corners, rounding every notch.

    max_pts=None -> no cap, return the DP vertices unchanged (lossless within tol_px).
    Returns (pts, achieved_tol, deviation_px)."""
    from shapely.geometry import LineString
    P = np.asarray(coords, float)
    if len(P) < 3:
        return P, tol_px, 0.0
    closed = np.allclose(P[0], P[-1])
    if max_pts is None or len(P) <= max_pts:
        return P, tol_px, 0.0
    lo, hi = tol_px, max(tol_px * 2, 1.0)
    ref = P
    L = LineString([tuple(p) for p in P])
    for _ in range(40):                       # find an upper tolerance that fits the budget
        if len(np.array(L.simplify(hi).coords)) <= max_pts:
            break
        hi *= 1.6
    for _ in range(24):                       # bisect down to the smallest tolerance that fits
        mid = 0.5 * (lo + hi)
        if len(np.array(L.simplify(mid).coords)) <= max_pts:
            hi = mid
        else:
            lo = mid
    Q = np.array(L.simplify(hi).coords)
    if closed and not np.allclose(Q[0], Q[-1]):
        Q = np.vstack([Q, Q[0]])
    return Q, hi, float(ef.max_deviation(ref, Q))


def label_slice_to_graph(label_slice, min_region_voxels=20, simplify_tol=0.75,
                         adaptive=True):
    """Trace boundaries from one integer label slice into a node/element graph where each
    element carries the TWO region codes it separates (shared borders are a SINGLE line).
    adaptive=True -> curvature-adaptive anchor placement (F1a). adaptive=False -> native
    traced/simplified vertices."""
    from scipy.ndimage import distance_transform_edt
    from shapely.geometry import LineString
    from shapely.ops import linemerge
    lab = label_slice
    H, W = lab.shape
    keep = np.zeros_like(lab)
    for rid in np.unique(lab):
        if rid == 0:
            continue
        m = lab == rid
        if m.sum() >= min_region_voxels:
            keep[m] = rid
    lab = keep

    from collections import defaultdict
    segs = defaultdict(list)
    for c in range(W - 1):
        diff = lab[:, c] != lab[:, c + 1]
        for r in np.where(diff)[0]:
            a, b = sorted((int(lab[r, c]), int(lab[r, c + 1])))
            segs[(a, b)].append([(c + 0.5, r - 0.5), (c + 0.5, r + 0.5)])
    for r in range(H - 1):
        diff = lab[r, :] != lab[r + 1, :]
        for c in np.where(diff)[0]:
            a, b = sorted((int(lab[r, c]), int(lab[r + 1, c])))
            segs[(a, b)].append([(c - 0.5, r + 0.5), (c + 0.5, r + 0.5)])

    g = bg.BoundaryGraph()          # local accumulator (was a module-level global; unsafe to reuse
                                    # now that add_node defers .nodes -- a fresh graph each call)
    n_pts, max_dev = 0, 0.0
    for (a, b), seglist in segs.items():
        merged = linemerge([LineString(s) for s in seglist])
        geoms = merged.geoms if merged.geom_type == "MultiLineString" else [merged]
        for geom in geoms:
            if SMOOTH_BOUNDARIES:
                # smooth FIRST (kills the staircase), THEN simplify ONCE at the normal tolerance.
                # DP now drops the redundant points along the smooth curve instead of re-finding
                # the step corners, because after smoothing there are no step corners left to find.
                pts = _smooth_ring(np.array(geom.coords), SMOOTH_SIGMA)
                if simplify_tol > 0 and len(pts) >= 2:
                    pts = np.array(LineString(pts).simplify(simplify_tol).coords)
            else:
                pts = np.array(geom.simplify(simplify_tol).coords)
            if adaptive and len(pts) >= 3 and TRACE_MAX_HANDLES:
                pts, _t, dev = _simplify_to_budget(pts, simplify_tol, TRACE_MAX_HANDLES)
                max_dev = max(max_dev, dev)            # cost of the handle cap, reported below
            if len(pts) < 2:
                continue
            n_pts += len(pts)
            prev = None
            for xy in pts:
                ni = g.add_node(xy)
                if prev is not None:
                    g.add_element(prev, ni, a, b)   # both codes set at creation
                prev = ni

    # The graph was built with a grid hash that DEFERS the .nodes array (add_node appends to a
    # pending list; it no longer re-vstacks on every insert, which was an O(n^2) build). Materialise
    # .nodes ONCE here, now the build loop is done.
    g.finalize_nodes()

    # ---- SEEDS: one deep interior point per region -----------------------------------
    # WHAT A SEED IS: for each region, the single voxel FARTHEST from that region's boundary --
    # the peak of the distance transform (distance_transform_edt gives every voxel its distance to
    # the nearest non-region voxel; argmax is the deepest point). So a seed is a point GUARANTEED
    # to lie well inside its region, one per region id, stored as ((x, y), region_id).
    #
    # WHY THE REBUILD NEEDS THEM: turning the fused lines back into filled regions (polygonize)
    # produces bare FACES with no labels. edge_fusion assigns each face an id mainly from the codes
    # its bounding arcs carry, which is exact for ordinary adjacent regions. But that rule is
    # PROVABLY UNDECIDABLE for an island or a donut hole: the face INSIDE a lone closed arc (a, i)
    # and the face OUTSIDE it share the identical code set {a, i}, so the codes alone cannot say
    # which side is the island. The seed breaks the tie -- faces partition the plane, so a region's
    # seed lands in exactly one face, and that face is the region. (In the legacy marching-line
    # identifier in bg_legacy.py the seed plays the same role from the other direction: the loop
    # walk STARTS at the seed and marches the boundary that encloses it. Same fact -- a known
    # interior point names the region -- used to grow a loop there, to label a face here.)
    seeds = []
    for rid in np.unique(lab):
        if rid == 0:
            continue
        m = lab == rid
        dt = distance_transform_edt(m)
        iy, ix = np.unravel_index(np.argmax(dt), dt.shape)   # deepest interior voxel
        seeds.append(((float(ix), float(iy)), int(rid)))
    if max_dev > 0:
        print(f"  !! TRACE_MAX_HANDLES={TRACE_MAX_HANDLES} cost up to {max_dev:.3f}px of "
              f"boundary deviation. That is a LOSSY ergonomics choice made BEFORE QA and "
              f"BEFORE fusion. Set it to None unless Illustrator is actually unusable.")
    return g, seeds


def _seed_points_on_slice(label_slice, min_region_voxels=6):
    """Seeds = one deep interior point per region (distance-transform maximum). CELL 7's
    rebuild uses them ONLY to disambiguate the faces the code-set rule PROVABLY cannot decide
    -- islands and donut holes, where the face inside a closed arc (a, i) and the face outside
    it have the SAME code set {a, i}. Everything else is decided by the codes alone."""
    from scipy.ndimage import distance_transform_edt
    seeds = []
    for rid in np.unique(label_slice):
        if rid == 0:
            continue
        m = label_slice == rid
        if m.sum() < min_region_voxels:
            continue
        dt = distance_transform_edt(m)
        iy, ix = np.unravel_index(np.argmax(dt), dt.shape)
        seeds.append(((float(ix), float(iy)), int(rid)))
    return seeds


def cell5_propagate_single_slice(min_region_voxels=6, simplify_tol=0.25):
    """Build the template boundary graph for SLICE_INDEX once, then give each subject an
    IDENTICAL copy (template-space design -> same slice, same labels for every subject).
    Writes one propagated .txt per subject into work_test/lines_propagated/."""
    regs = load_state("regs")
    lab = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    n_slices = lab.shape[SLICE_AXIS]
    if not (0 <= SLICE_INDEX < n_slices):
        raise IndexError(f"SLICE_INDEX={SLICE_INDEX} out of range 0..{n_slices-1}")

    sl = np.take(lab, SLICE_INDEX, axis=SLICE_AXIS)
    sl, midline_lr = _hemisphere_mask(sl, HEMISPHERE)
    n_regions = len(np.unique(sl)) - 1
    if (sl > 0).sum() < min_region_voxels:
        raise ValueError(f"slice {SLICE_INDEX} (hemisphere='{HEMISPHERE}') is essentially "
                         f"empty; pick another SLICE_INDEX or HEMISPHERE.")

    g, seeds = label_slice_to_graph(sl, min_region_voxels, simplify_tol, adaptive=True)
    template_graphs = {SLICE_INDEX: (g, seeds)}
    save_state("template_graphs", template_graphs)

    # give every subject an identical copy in template space (no per-subject warp needed)
    os.makedirs(f"{RUN_DIR}/lines_propagated", exist_ok=True)   # create-on-write
    out = {}
    for sid in regs:
        gc = bg.BoundaryGraph(nodes=np.array(g.nodes).copy(),
                              elements=[e[:] for e in g.elements])
        outp = f"{RUN_DIR}/lines_propagated/{sid}_slice_{SLICE_INDEX:03d}.txt"
        bg.write_ascii(gc, outp)
        out[sid] = {SLICE_INDEX: outp}
    save_state("propagated", out)

    save_state("test_meta", {"slice_index": SLICE_INDEX, "label_detail": LABEL_DETAIL,
                             "line_mode": LINE_MODE, "double_offset": LINE_DOUBLE_OFFSET,
                             "trace_max_handles": TRACE_MAX_HANDLES,
                             "simplify_tol": simplify_tol,
                             "hemisphere": HEMISPHERE, "midline_lr": midline_lr})

    # F1a diagnostic: how many points did the sagitta bound actually allocate, per arc?
    arcs = ef.graph_to_arcs(g)
    npts = [len(a.pts) for a in arcs]
    checks = [
        ("template slice has regions", n_regions > 0, "pick a slice with labels"),
        (f"slice {SLICE_INDEX} node count sane ({len(g.nodes)})",
         10 < len(g.nodes) < 60000, "adjust simplify_tol"),
        ("same slice+labels copied to every subject", len(out) == len(regs),
         "a subject is missing from regs"),
        ("handles per boundary are draggable in Illustrator",
         (max(npts) <= 200 if npts else True),
         f"max {max(npts) if npts else 0} handles on one boundary -> raise simplify_tol, or set "
         f"TRACE_MAX_HANDLES (lossy)"),
    ]
    report_outputs("CELL 5 propagate (single slice, template space)",
                   files=[list(v.values())[0] for v in out.values()],
                   state_keys=["template_graphs", "propagated", "test_meta"], checks=checks)
    print(f"\nPropagated slice {SLICE_INDEX} (hemisphere='{HEMISPHERE}') to {len(out)} subjects "
          f"({n_regions} regions, {len(g.nodes)} anchor nodes, {len(arcs)} arcs, "
          f"label_detail='{LABEL_DETAIL}', line_mode='{LINE_MODE}').")
    if npts:
        print(f"  Douglas-Peucker anchors (simplify_tol={simplify_tol}px): "
              f"{min(npts)}..{max(npts)} pts/arc, median {int(np.median(npts))}, "
              f"{sum(npts)} total.")
        print(f"  The old N_BOUNDARY_ANCHORS=24 would have written {24*len(arcs)} points, "
              f"EVENLY SPACED BY ARC LENGTH -- landing them BETWEEN the corners and rounding "
              f"every notch away, before the expert ever saw it.")
    if HEMISPHERE != "whole":
        print(f"  NOTE: single hemisphere kept (midline L-R row = {midline_lr:.1f}). "
              f"Run CELL 8 after QA/fusion to mirror it back to a full slice.")
    return out

#cell5_propagate_single_slice(min_region_voxels=6, simplify_tol=0.25)

In [ ]:
#@ IMPORTABLE
# =====================================================================================
# CELL 6  --  EXPORT ONE SLICE FOR QA   (locked BG/image layer + editable lines layer)
# =====================================================================================
# Same SVG format as before, with a clean LAYER split so the reviewer can only move the lines:
#   * Layer "background"  = white page rect + the T1 raster underlay. LOCKED.
#   * Layer "boundaries"  = the editable vector lines. UNLOCKED.
#   * Layer "labels"      = region annotation, emitted per LABEL_DETAIL.
#
# THREE CHANGES THAT MATTER, ALL OF THEM THINGS THAT WOULD OTHERWISE BREAK THE ROUND-TRIP:
#
#  1. THE BOUNDARIES LAYER NOW EMITS **ARCS**, NOT CODE-GROUP CHAINS.
#     The old _chain_boundaries() grouped every element by (matA, matB) and walked it. That
#     walk runs THROUGH triple points, so one emitted polyline could span several true arcs.
#     Smoothing then moved the junctions, and on re-import the arc/node topology was already
#     broken before fusion saw it. ef.graph_to_arcs() splits at genuine junctions (degree != 2,
#     or a third region arriving), which is the same decomposition the fusion consumes.
#
#  2. EACH ARC CARRIES ITS CODES IN THE ELEMENT **id**, NOT ONLY IN data-*.
#     ILLUSTRATOR STRIPS UNKNOWN data-* ATTRIBUTES ON SAVE. It keeps `id`. So the codes are
#     written as  id="bnd_{a}_{b}_{k}"  (and data-matA/matB as well, for Inkscape, which does
#     keep them). CELL 7's importer reads whichever survived. Without this the QA round-trip
#     silently loses every region code and the whole arc/node method collapses.
#
#  3. EXPORT_SMOOTH_SIGMA DEFAULTS TO 0.
#     The old SMOOTH_SIGMA = 3.0 Gaussian-smoothed every display polyline before writing it.
#     That is 60x FIT_TOL_PX and larger than KP_ALPHA -- it destroyed exactly the notches and
#     concavities the key-point machinery exists to preserve, it did so BEFORE the expert saw
#     them, and the smoothed curve was what came back on import. It also moved the lines off
#     the traced boundary, so the reviewer was correcting a curve that was not the data.
#     Set it back above 0 only if you want a cosmetic effect and accept that cost.
import base64, io, os, csv, colorsys, math
import numpy as np, nibabel as nib

# ---- display / labeling options -----------------------------------------------------
FLIP_LR              = False
LABEL_ALL_INSTANCES  = True
MIN_COMPONENT_VOXELS = 8
LABEL_FIT          = 0.60
LABEL_HEIGHT_FRAC  = 0.9
LABEL_FS_MAX       = 2.2
LABEL_FS_MIN       = 0.55
LABEL_OUTLINE_FRAC = 0.16

EXPORT_SMOOTH_SIGMA = 0.0   # <<< was SMOOTH_SIGMA = 3.0. See note 3 above. 0 = honest geometry.
RENDER_SMOOTH_ITERS = 2

# legend removed (abbreviations are drawn directly on regions); kept so old refs don't break.
LEGEND_GAP = 4.0; LEGEND_FS = 2.0; LEGEND_ROW = 2.6; LEGEND_SW = 2.0


def _fmt(v):
    """Coordinate formatter. Quantisation happens ON WRITE ONLY (AFAM F6 / SATM C8) -- there
    is none anywhere inside the fusion. grid_write=None -> full float."""
    p = FUSION_PARAMS
    if p.grid_write:
        v = round(float(v) / p.grid_write) * p.grid_write
    return f"{float(v):.{p.svg_decimals}f}"


# ---- number / color / name lookups (KEPT) -------------------------------------------
def _number_regions(anchors):
    rids = sorted({a[0] for a in anchors})
    return rids, {rid: i + 1 for i, rid in enumerate(rids)}

def region_color(rid):
    rid = int(rid)
    p = FUSION_PARAMS
    if rid >= p.unlabeled_id_start and rid < p.new_id_start:
        return (255, 0, 255)          # MAGENTA = an UNLABELED region (a QA gap). Unmissable.
    h = (rid * 0.61803398875) % 1.0; s, v = 0.65, 0.92
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return (round(r*255), round(g*255), round(b*255))

def _rgb_hex(rgb):
    return "#{:02x}{:02x}{:02x}".format(*rgb)

def load_region_table(lut_csv=None):
    lut_csv = lut_csv or TEMPLATE_LUT
    table = {}
    if not (lut_csv and os.path.exists(lut_csv)):
        print(f"  !! LUT not found at {lut_csv} -- labels fall back to numeric ids.")
        return table
    with open(lut_csv, newline="") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        print(f"  !! LUT {lut_csv} is empty."); return table
    cols = {c.lower().strip(): c for c in rows[0].keys()}
    id_col   = cols.get("id")
    abbr_col = cols.get("abbreviation") or cols.get("abbrev") or cols.get("label")
    name_col = cols.get("name") or cols.get("long_name") or cols.get("region")
    hex_col  = cols.get("color_hex") or cols.get("hex")
    for row in rows:
        try: rid = int(float(row.get(id_col)))
        except (TypeError, ValueError): continue
        abbrev = (row.get(abbr_col) if abbr_col else None) or str(rid)
        name   = (row.get(name_col) if name_col else None) or abbrev
        abbrev = str(abbrev).split(":")[-1].strip(); name = str(name).strip()
        rgb = region_color(rid)
        if hex_col and row.get(hex_col, "").startswith("#"):
            h = row[hex_col].lstrip("#")
            try: rgb = (int(h[0:2],16), int(h[2:4],16), int(h[4:6],16))
            except ValueError: pass
        table[rid] = {"abbrev": abbrev, "name": name, "rgb": rgb}
    return table

def _abbrev_for(table, rid):
    rid = int(rid)
    p = FUSION_PARAMS
    if rid == 0: return "bg"
    if p.unlabeled_id_start <= rid < p.new_id_start: return f"UNLAB{rid}"
    if rid >= p.new_id_start: return f"NEW{rid}"
    return table.get(rid, {}).get("abbrev", str(rid))

# ---- ORIENTATION transform (coronal) -- KEPT ----------------------------------------
def _orient_image(sl):
    img = sl.T
    img = img[::-1, :]
    if FLIP_LR: img = img[:, ::-1]
    return img

def _orient_xy(x_col, y_row, H_lr, W_si):
    nr = (W_si - 1) - x_col
    nc = (H_lr - 1 - y_row) if FLIP_LR else y_row
    return nc, nr

def _unorient_xy(x_disp, y_disp, H_lr, W_si):
    nc = (H_lr - 1 - x_disp) if FLIP_LR else x_disp
    x_col = (W_si - 1) - y_disp
    y_row = nc
    return x_col, y_row

def _orient_arr(P, H_lr, W_si):
    """Vectorised voxel -> display for an (n,2) array."""
    P = np.asarray(P, float)
    nr = (W_si - 1) - P[:, 0]
    nc = ((H_lr - 1) - P[:, 1]) if FLIP_LR else P[:, 1]
    return np.column_stack([nc, nr])

def _unorient_arr(D, H_lr, W_si):
    D = np.asarray(D, float)
    nc = ((H_lr - 1) - D[:, 0]) if FLIP_LR else D[:, 0]
    x_col = (W_si - 1) - D[:, 1]
    return np.column_stack([x_col, nc])

def _slice_to_png_bytes(t1_volume, slice_index, axis,
                        hemisphere="whole", midline_lr=None):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    raw = np.take(t1_volume, slice_index, axis=axis).astype(np.float32)
    img = _orient_image(raw)               # shape (H, W); W = raw axis-0 (L-R)
    lo, hi = np.percentile(img[img > 0], (1, 99)) if (img > 0).any() else (0, 1)
    img = np.clip((img - lo) / (hi - lo + 1e-9), 0, 1)

    if hemisphere != "whole" and midline_lr is not None:
        W_img = img.shape[1]
        mid_col = int(round(midline_lr))
        keep_low = (hemisphere == "left")
        if FLIP_LR:
            keep_low = not keep_low
            mid_col = W_img - mid_col
        if keep_low:
            img[:, mid_col:] = 0.0
        else:
            img[:, :mid_col] = 0.0

    buf = io.BytesIO(); h, w = img.shape
    fig = plt.figure(figsize=(w/100, h/100), dpi=100); ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(img, cmap="gray", origin="upper"); ax.axis("off")
    fig.savefig(buf, format="png", dpi=100); plt.close(fig)
    return buf.getvalue(), w, h, raw.shape

# ---- label placement (KEPT; eats (rid, x, y, clearance, n_instances)) ----------------
def _label_points_on_slice(label_slice, label_all_instances=None, min_component_voxels=None):
    from scipy.ndimage import distance_transform_edt, label as cc_label
    if label_all_instances is None: label_all_instances = LABEL_ALL_INSTANCES
    if min_component_voxels is None: min_component_voxels = MIN_COMPONENT_VOXELS
    anchors = []
    for rid in np.unique(label_slice):
        if rid == 0: continue
        comps, n = cc_label(label_slice == rid)
        allc = []
        for ci in range(1, n + 1):
            comp = comps == ci
            dt = distance_transform_edt(comp)
            iy, ix = np.unravel_index(np.argmax(dt), dt.shape)
            allc.append((int(rid), float(ix), float(iy), float(dt[iy, ix]), int(comp.sum())))
        if not allc: continue
        kept = [c for c in allc if c[4] >= min_component_voxels]
        if not kept: kept = [max(allc, key=lambda t: t[4])]
        ninst = len(kept)
        if label_all_instances:
            anchors.extend([(r, x, y, c, ninst) for (r, x, y, c, _s) in kept])
        else:
            r, x, y, c, _s = max(kept, key=lambda t: t[3])
            anchors.append((r, x, y, c, ninst))
    return anchors

def _place_labels(anchors, label_text_of, H_lr, W_si, W, H):
    from collections import defaultdict
    by_region = defaultdict(list)
    for rid, x, y, clearance, _ninst in anchors:
        by_region[rid].append((x, y, clearance))
    labels = []
    for rid, comps in by_region.items():
        txt = str(label_text_of.get(rid, rid)); ndig = max(len(txt), 1)
        for (x, y, clearance) in comps:
            d = 2.0 * clearance
            fs = min((d * 0.9) / (ndig * LABEL_FIT), d * LABEL_HEIGHT_FRAC)
            fs = min(max(fs, LABEL_FS_MIN), LABEL_FS_MAX)
            xd, yd = _orient_xy(x, y, H_lr, W_si)
            labels.append({"rid": rid, "cx": xd, "cy": yd, "fs": fs, "txt": txt})
    return labels


# =====================================================================================
# SVG READER  --  the QA round-trip.  THIS IS WHERE ILLUSTRATOR BREAKS THINGS.
# =====================================================================================
# The old read_svg_graph() read <polyline points="..."> and took the codes from
# data-matA / data-matB. Both assumptions fail on a real Illustrator round-trip:
#
#   * ILLUSTRATOR CONVERTS <polyline> TO <path d="M ... L ... Z">.
#     -> a polyline-only reader silently returns an EMPTY graph.
#   * ILLUSTRATOR STRIPS UNKNOWN data-* ATTRIBUTES on save.
#     -> the region codes vanish and every element becomes (0, 0).
#   * ILLUSTRATOR ESCAPES id CHARACTERS as _xHH_ (an underscore becomes _x5F_).
#   * ILLUSTRATOR / INKSCAPE MAY WRAP LAYERS IN A transform="matrix(...)".
#     -> every coordinate is offset by a constant the reader never knew about.
#
# So: read BOTH element types, take the codes from data-* IF PRESENT and fall back to the
# id (which survives), un-escape _xHH_, and compose any ancestor transforms.
import xml.etree.ElementTree as ET
import re as _re


def _svg_unescape(s):
    """Illustrator writes _x5F_ for '_', _x2D_ for '-', etc. Undo it."""
    return _re.sub(r"_x([0-9A-Fa-f]{2})_", lambda m: chr(int(m.group(1), 16)), s or "")

from xml.sax.saxutils import escape as _xml_escape  # stdlib: turns & < > into &amp; &lt; &gt;


def _parse_transform(t):
    """Compose an SVG transform string into a 2x3 affine [[a,c,e],[b,d,f]]."""
    M = np.array([[1., 0., 0.], [0., 1., 0.]])
    if not t:
        return M
    for name, args in _re.findall(r"(\w+)\s*\(([^)]*)\)", t):
        v = [float(x) for x in _re.split(r"[\s,]+", args.strip()) if x]
        if name == "matrix" and len(v) == 6:
            N = np.array([[v[0], v[2], v[4]], [v[1], v[3], v[5]]])
        elif name == "translate":
            N = np.array([[1., 0., v[0]], [0., 1., v[1] if len(v) > 1 else 0.]])
        elif name == "scale":
            sx = v[0]; sy = v[1] if len(v) > 1 else v[0]
            N = np.array([[sx, 0., 0.], [0., sy, 0.]])
        elif name == "rotate" and len(v) >= 1:
            a = math.radians(v[0]); c, s = math.cos(a), math.sin(a)
            N = np.array([[c, -s, 0.], [s, c, 0.]])
            if len(v) == 3:
                T1 = np.array([[1., 0., v[1]], [0., 1., v[2]]])
                T2 = np.array([[1., 0., -v[1]], [0., 1., -v[2]]])
                N = _mm(_mm(T1, N), T2)
        else:
            continue
        M = _mm(M, N)
    return M


def _mm(A, B):
    A3 = np.vstack([A, [0., 0., 1.]]); B3 = np.vstack([B, [0., 0., 1.]])
    return (A3 @ B3)[:2]


def _apply_tf(M, P):
    P = np.asarray(P, float)
    return np.column_stack([M[0, 0]*P[:, 0] + M[0, 1]*P[:, 1] + M[0, 2],
                            M[1, 0]*P[:, 0] + M[1, 1]*P[:, 1] + M[1, 2]])


def _flatten_cubic(p0, p1, p2, p3, tol):
    """Adaptive cubic flattening. Splits until the control polygon is within tol of the chord."""
    d1 = np.abs(np.cross(p3 - p0, p0 - p1)); d2 = np.abs(np.cross(p3 - p0, p0 - p2))
    L = np.hypot(*(p3 - p0)) + 1e-12
    n = max(2, int(math.ceil(math.sqrt(max(d1, d2) / L / max(tol, 1e-6) * 3.0))) + 1)
    n = min(n, 64)
    t = np.linspace(0, 1, n)[1:, None]
    return ((1-t)**3)*p0 + 3*((1-t)**2)*t*p1 + 3*(1-t)*(t**2)*p2 + (t**3)*p3


def _parse_path_d(d, tol):
    """SVG 'd' -> [(pts, closed), ...]. Handles M/L/H/V/C/S/Q/T/Z, absolute and relative.
    Beziers are flattened at `tol` (== FUSION_PARAMS.flatten_px, which AFAM F6 requires to be
    <= fit_tol_px -- otherwise you destroy the curve, then find key points on the wreckage,
    then carefully sample the wreckage to 0.05px)."""
    toks = _re.findall(r"([MmLlHhVvCcSsQqTtZz])|(-?\d*\.?\d+(?:[eE][-+]?\d+)?)", d or "")
    items = [(a or b) for a, b in toks]
    subs, pts, cur, start, cmd, prev_c2, prev_q = [], [], np.zeros(2), np.zeros(2), None, None, None
    i = 0
    def _num():
        nonlocal i
        v = float(items[i]); i += 1; return v
    while i < len(items):
        if items[i] in "MmLlHhVvCcSsQqTtZz":
            cmd = items[i]; i += 1
        if cmd is None:
            i += 1; continue
        rel = cmd.islower(); C = cmd.upper()
        if C == "M":
            if pts and len(pts) >= 2:
                subs.append((np.array(pts), False))
            x, y = _num(), _num()
            cur = (cur + [x, y]) if rel else np.array([x, y])
            start = cur.copy(); pts = [cur.copy()]
            cmd = "l" if rel else "L"
        elif C == "L":
            x, y = _num(), _num()
            cur = (cur + [x, y]) if rel else np.array([x, y])
            pts.append(cur.copy())
        elif C == "H":
            x = _num(); cur = np.array([cur[0] + x, cur[1]]) if rel else np.array([x, cur[1]])
            pts.append(cur.copy())
        elif C == "V":
            y = _num(); cur = np.array([cur[0], cur[1] + y]) if rel else np.array([cur[0], y])
            pts.append(cur.copy())
        elif C in ("C", "S"):
            if C == "C":
                c1 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            else:
                c1 = 2*cur - prev_c2 if prev_c2 is not None else cur.copy()
            c2 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            p3 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            pts.extend(_flatten_cubic(cur, c1, c2, p3, tol))
            prev_c2 = c2; cur = p3; prev_q = None
        elif C in ("Q", "T"):
            if C == "Q":
                q = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            else:
                q = 2*cur - prev_q if prev_q is not None else cur.copy()
            p2 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            c1 = cur + 2.0/3.0*(q - cur); c2 = p2 + 2.0/3.0*(q - p2)
            pts.extend(_flatten_cubic(cur, c1, c2, p2, tol))
            prev_q = q; cur = p2; prev_c2 = None
        elif C == "Z":
            if len(pts) >= 2:
                subs.append((np.array(pts), True))
            pts = [start.copy()]; cur = start.copy()
            cmd = None
        else:
            i += 1
        if C not in ("C", "S"): prev_c2 = None
        if C not in ("Q", "T"): prev_q = None
    if len(pts) >= 2:
        subs.append((np.array(pts), False))
    return subs


def _meta_from_el(el):
    """(codes, arc_index, side) from data-* if the editor kept them, else from the id.

    THE ARC INDEX MATTERS. Several arcs legitimately share the same code pair -- e.g. the
    (0, 11) border reaches the brain surface in two separate places, so there are TWO (0,11)
    arcs. Collapsing double-lines by CODE ALONE would then average one arc's +side against a
    DIFFERENT arc's -side. (Measured: 5.2px of garbage.) The id carries the arc index precisely
    so the pair can be reunited: bnd_{a}_{b}_{k}[_s{side}]."""
    eid = _svg_unescape(el.get("id", ""))
    m = _re.match(r"^bnd_(-?\d+)_(-?\d+)_(\d+)(?:_s(\d+))?$", eid)
    code = arc = side = None
    if m:
        code = (int(m.group(1)), int(m.group(2)))
        arc  = int(m.group(3))
        side = int(m.group(4)) if m.group(4) is not None else None
    a, b = el.get("data-matA"), el.get("data-matB")
    if a is not None and b is not None:
        try: code = (int(a), int(b))
        except ValueError: pass
    if el.get("data-arc") is not None:
        try: arc = int(el.get("data-arc"))
        except ValueError: pass
    return code, arc, side


def _project_onto(P, R):
    """Closest point on polyline R for each point of P. Vectorised point-to-segment."""
    P = np.asarray(P, float); R = np.asarray(R, float)
    A, B = R[:-1], R[1:]
    AB = B - A
    L2 = (AB ** 2).sum(1) + 1e-12
    t = np.clip(((P[:, None, :] - A[None]) * AB[None]).sum(2) / L2[None], 0.0, 1.0)
    proj = A[None] + t[..., None] * AB[None]                 # (n, m, 2)
    d = np.linalg.norm(P[:, None, :] - proj, axis=2)         # (n, m)
    return proj[np.arange(len(P)), d.argmin(axis=1)]


def _collapse_double_lines(items):
    """LINE_MODE='double' writes TWO half-lines per border, offset +/- double_offset from the
    true boundary. Re-importing both would give two parallel arcs ~2*offset apart carrying the
    SAME codes. The fusion would treat them as two DIFFERENT boundaries between the same pair of
    regions, and polygonize would turn the sliver between them into a spurious face. Average the
    pair back onto the true border.

    Keyed by (code, ARC INDEX, closed). The arc index is essential: several arcs legitimately
    share a code pair (the (0,11) border reaches the brain surface in two places, so there are
    TWO (0,11) arcs), and collapsing by code alone would average one arc's +side against a
    DIFFERENT arc's -side. Measured: 5.2px of garbage.

    THE AVERAGE IS VERTEX-WISE, NOT ARC-LENGTH-WISE. _offset_polyline maps vertex i of the true
    border to vertex i of BOTH half-lines, so vertex i of one pairs with vertex i of the other,
    exactly. Resampling both to a common n by ARC LENGTH would NOT work: the outer half-line is
    longer than the inner one (it bulges at every corner), so equal arc-length fractions land on
    DIFFERENT parts of the border -- by up to 10% of the arc on a tight island. Measured: also
    5.2px of garbage. If the expert added or removed handles on one side the counts no longer
    match, and we fall back to a nearest-point projection, which does not care about arc length."""
    from collections import defaultdict
    by = defaultdict(list)
    for (code, arc, side, P, closed) in items:
        by[(code, arc, closed)].append((P, side))
    out, n_pairs = [], 0
    for (code, arc, closed), group in by.items():
        sides = {s for _, s in group if s is not None}
        if len(group) == 2 and len(sides) == 2:
            A, B = group[0][0], group[1][0]
            if len(A) == len(B):                      # untouched: vertex i <-> vertex i. Exact.
                if np.sum((A - B) ** 2) > np.sum((A - B[::-1]) ** 2):
                    B = B[::-1]
                M = 0.5 * (A + B)
            else:                                     # edited: project instead
                M = 0.5 * (A + _project_onto(A, B))
            out.append((code, M, closed))
            n_pairs += 1
        else:
            for P, _ in group:
                out.append((code, P, closed))
    return out, n_pairs


def read_qa_svg(svg_path, H_lr, W_si, verbose=True, line_mode=None, double_offset=None,
                params=None):
    """Parse a QA-edited SVG (Inkscape OR Illustrator) back into a BoundaryGraph carrying BOTH
    region codes on every element.

    A closed path with no parseable code is a region the EXPERT DREW: it gets a fresh id from
    params.new_id_start upward, coded (new_id, OUTER). It is reported, never silently dropped."""
    params = params or FUSION_PARAMS
    meta = load_state("test_meta", default={})
    line_mode     = line_mode     or meta.get("line_mode", "single")
    double_offset = double_offset if double_offset is not None else meta.get("double_offset", 0.35)

    tree = ET.parse(svg_path); root = tree.getroot()
    parent = {c: p for p in root.iter() for c in p}

    def _lname(t): return t.rsplit("}", 1)[-1]
    def _tf_chain(el):
        M, node = np.array([[1., 0., 0.], [0., 1., 0.]]), el
        chain = []
        while node is not None:
            chain.append(node.get("transform"))
            node = parent.get(node)
        for t in reversed(chain):
            if t: M = _mm(M, _parse_transform(t))
        return M

    items, n_new, n_nocode, n_path, n_poly, n_pairs = [], 0, 0, 0, 0, 0
    for el in root.iter():
        tag = _lname(el.tag)
        if tag not in ("polyline", "path", "polygon"):
            continue
        # skip the atlas-face layer if it is present (those are fills, not boundaries)
        if (el.get("id") or "").startswith("face_"):
            continue
        M = _tf_chain(el)

        subs = []
        if tag in ("polyline", "polygon"):
            raw = (el.get("points") or "").strip()
            if not raw:
                continue
            v = [float(t) for t in _re.split(r"[\s,]+", raw) if t]
            if len(v) < 4:
                continue
            subs = [(np.array(v, float).reshape(-1, 2), tag == "polygon")]
            n_poly += 1
        else:
            subs = _parse_path_d(el.get("d", ""), params.flatten_px)
            n_path += 1
        if not subs:
            continue

        code, arc_i, side = _meta_from_el(el)
        for si, (P, closed) in enumerate(subs):
            if len(P) < 2:
                continue
            D = _apply_tf(M, P)
            vox = _unorient_arr(D, H_lr, W_si)         # display -> voxel
            if code is None:
                # No code anywhere. A CLOSED shape is a region the EXPERT DREW -> new id.
                # An OPEN one is a stray stroke: it cannot bound anything, so it is ignored
                # (and counted, so it is never silently swallowed).
                n_nocode += 1
                if closed or (len(vox) > 3 and np.hypot(*(vox[0] - vox[-1])) < 1.0):
                    items.append((None, None, None, vox, True))
                continue
            items.append((tuple(sorted(code)), (arc_i, si), side, vox, bool(closed)))

    coded   = [it for it in items if it[0] is not None]
    uncoded = [(None, P, True) for (c, a, s, P, cl) in items if c is None]
    if line_mode == "double":
        collapsed, n_pairs = _collapse_double_lines(coded)
    else:
        collapsed = [(c, P, cl) for (c, a, s, P, cl) in coded]
    collapsed += uncoded

    # -- expert-drawn closed paths with no code -> NEW regions -------------------------
    reg = ef.load_registry(REGISTRY_PATH)
    next_new = ef.registry_next_new(reg, params)
    final = []
    for (code, P, closed) in collapsed:
        if code is None:
            code = (params.outer_code, next_new)
            next_new += 1
            n_new += 1
        final.append((tuple(sorted(code)), P, closed))

    g = bg.BoundaryGraph(nodes=np.empty((0, 2)), elements=[])
    index, xy = {}, []
    def _nid(p):
        k = (round(float(p[0]), 6), round(float(p[1]), 6))
        if k not in index:
            index[k] = len(xy); xy.append((float(p[0]), float(p[1])))
        return index[k]
    for (a, b), P, closed in final:
        Q = np.vstack([P, P[0]]) if (closed and np.hypot(*(P[0] - P[-1])) > 1e-9) else P
        prev = None
        for p in Q:
            ni = _nid(p)
            if prev is not None and prev != ni:
                g.elements.append([prev, ni, int(a), int(b)])
            prev = ni
    g.nodes = np.asarray(xy, float) if xy else np.zeros((0, 2))

    if verbose:
        print(f"    {os.path.basename(svg_path)}: {n_poly} polyline(s) + {n_path} path(s) -> "
              f"{len(g.elements)} elements, {len(ef.graph_to_arcs(g))} arcs"
              + (f", {n_pairs} double-line pair(s) collapsed" if n_pairs else "")
              + (f", {n_new} NEW expert region(s) from id {params.new_id_start}" if n_new else "")
              + (f", {n_nocode} open shape(s) with no code IGNORED" if n_nocode and not n_new else ""))
        if n_path and not n_poly:
            print(f"      (this file is Illustrator-saved: <polyline> became <path>, and the "
                  f"codes were recovered from id='bnd_A_B_k' because data-* was stripped)")
    return g

# ---- smoothing / offsetting (KEPT, but smoothing is now OFF by default) --------------
def _smooth_contour(pts, closed, sigma):
    """Gaussian-smooth a display polyline. DEFAULT sigma IS NOW 0 -- see note 3 at the top of
    this cell. Endpoints are pinned so junctions stay put, but every INTERIOR vertex moves,
    which is precisely what destroys a notch."""
    from scipy.ndimage import gaussian_filter1d
    P = np.asarray(pts, float)
    if len(P) < 4 or sigma <= 0: return P
    if closed and np.allclose(P[0], P[-1]): P = P[:-1]
    ring = np.vstack([P, P[0]]) if closed else P
    seg = np.sqrt((np.diff(ring, axis=0) ** 2).sum(1)); d = np.r_[0, np.cumsum(seg)]
    if d[-1] <= 0: return P
    u = np.arange(0, d[-1], 1.0)
    xs = np.interp(u, d, ring[:, 0]); ys = np.interp(u, d, ring[:, 1])
    if closed:
        xs = gaussian_filter1d(xs, sigma, mode="wrap"); ys = gaussian_filter1d(ys, sigma, mode="wrap")
        out = np.c_[xs, ys]; return np.vstack([out, out[0]])
    else:
        end0, end1 = ring[0].copy(), ring[-1].copy()
        xs = gaussian_filter1d(xs, sigma, mode="nearest"); ys = gaussian_filter1d(ys, sigma, mode="nearest")
        out = np.c_[xs, ys]; out[0], out[-1] = end0, end1
        return out

def _offset_polyline(disp, offset):
    """Offset an ordered display polyline by `offset` px along its per-vertex normal.
    Used to build the two half-lines in double mode. CELL 7 collapses them back."""
    P = np.asarray(disp, float)
    if len(P) < 2: return P
    closed = np.allclose(P[0], P[-1])
    Q = P[:-1] if closed else P
    n = len(Q)
    tang = np.zeros_like(Q)
    for i in range(n):
        a = Q[i-1] if (closed or i > 0) else Q[i]
        b = Q[(i+1) % n] if (closed or i < n-1) else Q[i]
        t = b - a; nrm = np.hypot(*t)
        tang[i] = t / nrm if nrm > 1e-9 else np.array([1.0, 0.0])
    normal = np.c_[-tang[:, 1], tang[:, 0]]
    out = Q + offset * normal
    if closed: out = np.vstack([out, out[0]])
    return out


# ---- SVG writer: ARC-BASED boundaries layer, codes in the id ------------------------
def _graph_to_svg(g, path, t1_volume=None, slice_index=None, axis=2,
                  label_anchors=None, table=None, stroke=1.2,
                  line_mode="single", double_offset=0.35, label_detail="abbrev",
                  hemisphere="whole", midline_lr=None, smooth_sigma=None,
                  regions=None, fill_opacity=0.0):
    """Write one slice to SVG.

    regions={rid: [shapely Polygon]} -> ALSO draw the filled polygon atlas underneath the
    lines (CELL 7's rebuild output). fill_opacity=0 keeps it invisible; raise it to SEE the
    atlas, including any magenta UNLABELED faces where the QA left a gap.
    smooth_sigma=None -> EXPORT_SMOOTH_SIGMA (0 by default; do not raise it for QA export)."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)    # create-on-write
    if not len(g.nodes) or t1_volume is None or slice_index is None:
        open(path, "w").write("<svg xmlns='http://www.w3.org/2000/svg'/>"); return
    table = table or {}
    sigma = EXPORT_SMOOTH_SIGMA if smooth_sigma is None else smooth_sigma
    png, W, H, (H_lr, W_si) = _slice_to_png_bytes(
        t1_volume, slice_index, axis, hemisphere=hemisphere, midline_lr=midline_lr)
    b64 = base64.b64encode(png).decode("ascii")

    show_text = (label_detail != "none")
    region_ids, _num = _number_regions(label_anchors or [])
    abbrev_of = {rid: _abbrev_for(table, rid) for rid in region_ids}
    labels = _place_labels(label_anchors, abbrev_of, H_lr, W_si, W, H) \
             if (label_anchors and show_text) else []

    total_w, total_h = W, H
    out = ['<?xml version="1.0" encoding="UTF-8"?>',
           f'<svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" '
           f'xmlns:inkscape="http://www.inkscape.org/namespaces/inkscape" '
           f'xmlns:sodipodi="http://sodipodi.sourceforge.net/DTD/sodipodi-0.0.dtd" '
           f'viewBox="0 0 {total_w:.1f} {total_h:.1f}" width="{total_w:.1f}" height="{total_h:.1f}">']

    # -- LAYER 1: background (page + T1 raster). LOCKED --------------------------------
    out.append('<g inkscape:groupmode="layer" inkscape:label="background" id="layer_background" '
               'sodipodi:insensitive="true" style="pointer-events:none">')
    out.append(f'<rect x="0" y="0" width="{total_w:.1f}" height="{total_h:.1f}" fill="white"/>')
    out.append(f'<image x="0" y="0" width="{W}" height="{H}" '
               f'xlink:href="data:image/png;base64,{b64}" style="pointer-events:none"/>')
    out.append('</g>')

    # -- LAYER 1b: the POLYGON ATLAS (CELL 7's rebuild). LOCKED. ------------------------
    # THE POLYGONS ARE THE ATLAS. Faces get their region id by INTERSECTING the codes of the
    # arcs on their boundary; whatever that rule cannot decide is an explicit UNLABELED region,
    # drawn MAGENTA, so a gap in the QA shows up ON the atlas instead of vanishing.
    if regions and fill_opacity > 0:
        out.append('<g inkscape:groupmode="layer" inkscape:label="atlas_faces" id="layer_faces" '
                   f'sodipodi:insensitive="true" style="pointer-events:none" '
                   f'opacity="{fill_opacity:.2f}">')
        for rid, faces in sorted(regions.items()):
            col = _rgb_hex(region_color(rid))
            for k, poly in enumerate(faces):
                rings = [np.asarray(poly.exterior.coords, float)] + \
                        [np.asarray(r.coords, float) for r in poly.interiors]
                d = []
                for R in rings:                       # even-odd fill -> holes are real holes
                    D = _orient_arr(R, H_lr, W_si)
                    d.append("M " + " L ".join(f"{_fmt(x)},{_fmt(y)}" for x, y in D) + " Z")
                out.append(f'<path d="{" ".join(d)}" fill="{col}" fill-rule="evenodd" '
                           f'stroke="none" id="face_{int(rid)}_{k}" data-region="{int(rid)}"/>')
        out.append('</g>')

    # -- LAYER 2: boundaries (editable). ONE POLYLINE PER **ARC**. UNLOCKED. ------------
    out.append('<g inkscape:groupmode="layer" inkscape:label="boundaries" id="layer_boundaries" '
               'fill="none" stroke-linejoin="round" stroke-linecap="round">')
    for k, arc in enumerate(ef.graph_to_arcs(g)):
        mA, mB = int(arc.code[0]), int(arc.code[1])
        P = np.vstack([arc.pts, arc.pts[0]]) if arc.closed else arc.pts
        disp = _orient_arr(P, H_lr, W_si)
        if len(disp) < 2:
            continue
        if sigma > 0:
            disp = _smooth_contour(disp, bool(arc.closed), sigma)
        na, nb = _abbrev_for(table, mA), _abbrev_for(table, mB)
        # id carries the codes. Illustrator STRIPS data-* but KEEPS id -- this is the only
        # channel that survives an Illustrator round-trip.
        eid = f"bnd_{mA}_{mB}_{k}"
        if line_mode == "double":
            half = stroke / 2.0
            colA = _rgb_hex(region_color(mA)) if mA > 0 else "#888888"
            colB = _rgb_hex(region_color(mB)) if mB > 0 else "#888888"
            out.append(f'<g id="arc_{eid}" data-arc="{k}">')
            for si, (sgn, col, mat) in enumerate(((+1.0, colA, mA), (-1.0, colB, mB))):
                od = _offset_polyline(disp, sgn * double_offset)
                pts_str = " ".join(f"{_fmt(x)},{_fmt(y)}" for (x, y) in od)
                out.append(f'<polyline id="{eid}_s{si}" points="{pts_str}" stroke="{col}" '
                           f'stroke-width="{half:.3f}" fill="none" data-matA="{mA}" '
                           f'data-matB="{mB}" data-side="{mat}" data-arc="{k}">'
                           f'<title>{_xml_escape(str(na))} | {_xml_escape(str(nb))}</title></polyline>')
            out.append('</g>')
        else:
            rid = mA if mA > 0 else mB
            col = _rgb_hex(region_color(rid)) if rid > 0 else "#888888"
            pts_str = " ".join(f"{_fmt(x)},{_fmt(y)}" for (x, y) in disp)
            out.append(f'<polyline id="{eid}" points="{pts_str}" stroke="{col}" '
                       f'stroke-width="{stroke}" fill="none" data-matA="{mA}" '
                       f'data-matB="{mB}" data-arc="{k}">'
                       f'<title>{_xml_escape(str(na))} | {_xml_escape(str(nb))}</title></polyline>')
    out.append('</g>')

    # -- LAYER 3: labels ---------------------------------------------------------------
    if labels:
        out.append('<g inkscape:groupmode="layer" inkscape:label="labels" id="layer_labels" '
                   'font-family="sans-serif" font-weight="bold" '
                   'text-anchor="middle" dominant-baseline="central">')
        _label_seen = {}
        for L in labels:
            col = _rgb_hex(region_color(L["rid"]))
            pos = f'x="{L["cx"]:.2f}" y="{L["cy"]:.2f}" font-size="{L["fs"]:.2f}"'
            _n = _label_seen.get(L["rid"], 0); _label_seen[L["rid"]] = _n + 1
            out.append(f'<g id="label_{int(L["rid"])}_{_n}" data-region="{L["rid"]}">')
            _txt = _xml_escape(str(L["txt"]))
            out.append(f'<text {pos} fill="black" stroke="black" '
                       f'stroke-width="{L["fs"]*LABEL_OUTLINE_FRAC:.3f}" '
                       f'stroke-linejoin="round">{_xml_escape(str(L["txt"]))}</text>')
            out.append(f'<text {pos} fill="{col}" data-region="{L["rid"]}">{_xml_escape(str(L["txt"]))}</text>')
            out.append('</g>')
        out.append('</g>')   # FIX: close the "labels" layer group opened above; without this the

    out.append('</svg>')
    _doc = "\n".join(out)
    # TEMP DEBUG — validate the SVG we just built and point at the exact bad line
    try:
        import xml.etree.ElementTree as _ET
        _ET.fromstring(_doc)
    except _ET.ParseError as _e:
        _ln = getattr(_e, "position", (0, 0))[0]
        _lines = _doc.split("\n")
        _lo, _hi = max(0, _ln - 3), min(len(_lines), _ln + 2)
        print(f"[svg-debug] {path} is malformed: {_e}")
        for _j in range(_lo, _hi):
            print(f"[svg-debug]   {_j+1}: {_lines[_j]}")
    open(path, "w").write(_doc)


# ---- the cell: export just the toggled slice for each subject -----------------------
def cell6_export_single_slice(stroke=0.36):
    base = f"{RUN_DIR}/lines_export_for_qa"
    slices_dir, tables_dir = f"{base}/slices", f"{base}/tables"
    os.makedirs(slices_dir, exist_ok=True); os.makedirs(tables_dir, exist_ok=True)

    prop = load_state("propagated")
    regs = load_state("regs")
    meta = load_state("test_meta")
    table = load_region_table(TEMPLATE_LUT)
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    sidx = meta["slice_index"]
    line_mode = meta["line_mode"]; dbl = meta["double_offset"]; detail = meta["label_detail"]

    first_sid = next(iter(prop))
    written = []
    for sid, slices in prop.items():
        t1_vol = nib.load(regs[sid]["warped"]).get_fdata() if sid in regs else None
        g = bg.read_ascii(slices[sidx])
        lab_slice = np.take(lab_vol, sidx, axis=SLICE_AXIS)
        lab_slice, _ = _hemisphere_mask(lab_slice, meta["hemisphere"])  # FIX: mask labels to kept hemisphere
        anchors = _label_points_on_slice(lab_slice)
        svg = f"{slices_dir}/{sid}_slice_{sidx:03d}.svg"
        _graph_to_svg(g, svg, t1_volume=t1_vol, slice_index=sidx, axis=SLICE_AXIS,
                      label_anchors=anchors, table=table, stroke=stroke,
                      line_mode=line_mode, double_offset=dbl, label_detail=detail,
                      hemisphere=meta["hemisphere"], midline_lr=meta["midline_lr"])
        written.append(svg)
        # AUTO-QA: drop an identical copy into the fusion input dir (qc/) so a run can go
        # export -> fuse with no manual copy. lines_export_for_qa/ keeps the ORIGINAL reference.
        import shutil
        os.makedirs(f"{RUN_DIR}/qc", exist_ok=True)             # create-on-write
        qa_dst = f"{RUN_DIR}/qc/{sid}_slice_{sidx:03d}.svg"
        shutil.copyfile(svg, qa_dst)
        print(f"  [auto-QA] {sid}: copied export -> {qa_dst}")   # TROUBLESHOOTING PRINT
        if sid == first_sid and detail != "none":
            region_ids, _ = _number_regions(anchors)
            with open(f"{tables_dir}/slice_{sidx:03d}_regions.csv", "w", newline="") as f:
                w = csv.writer(f)
                w.writerow(["abbreviation", "name", "id", "color_hex", "type"])
                for rid in region_ids:
                    info = table.get(rid, {})
                    ab = info.get("abbrev", str(rid)); nm = info.get("name", ab)
                    col = _rgb_hex(info.get("rgb", region_color(rid)))
                    kind = combined_lut.get(rid, {}).get("kind", "region")
                    w.writerow([ab, nm, rid, col, kind])

    # round-trip check: re-import what we just wrote and confirm the codes survived
    g0 = bg.read_ascii(prop[first_sid][sidx])
    H_lr, W_si = np.take(lab_vol, sidx, axis=SLICE_AXIS).shape
    g_rt = read_qa_svg(written[0], H_lr, W_si, verbose=False)
    codes_out = {tuple(sorted((int(e[2]), int(e[3])))) for e in g0.elements}
    codes_in  = {tuple(sorted((int(e[2]), int(e[3])))) for e in g_rt.elements}
    d_rt = ef._curve_dist(ef.graph_to_arcs(g_rt), ef.graph_to_arcs(g0), 0.5)

    checks = [
        ("every region code survives the SVG round-trip", codes_out <= codes_in,
         f"missing on re-import: {sorted(codes_out - codes_in)[:6]}"),
        (f"round-trip geometry error {d_rt:.4f}px <= 2x grid",
         d_rt <= max(2 * (FUSION_PARAMS.grid_write or 0.01), 0.05),
         "EXPORT_SMOOTH_SIGMA > 0, or the display transform is not inverting"),
    ]
    report_outputs("CELL 6 export (single slice)", files=written, checks=checks)
    print(f"\nExported slice {sidx} for {len(written)} subjects to {slices_dir}")
    print(f"  line_mode='{line_mode}'  label_detail='{detail}'  "
          f"hemisphere='{meta.get('hemisphere','whole')}'  smooth_sigma={EXPORT_SMOOTH_SIGMA}")
    print(f"  {len(ef.graph_to_arcs(g0))} ARCS written, codes carried in id='bnd_A_B_k' "
          f"(Illustrator strips data-*, keeps id).")
    return written


def preview_qa_slice(sid=None, stroke=0.4, scale=2.0):
    """Render the toggled slice for one subject inline in Colab."""
    from IPython.display import HTML, display
    import tempfile
    prop = load_state("propagated"); regs = load_state("regs"); meta = load_state("test_meta")
    table = load_region_table(TEMPLATE_LUT)
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    sid = sid or next(iter(prop)); sidx = meta["slice_index"]
    g = bg.read_ascii(prop[sid][sidx])
    t1_vol = nib.load(regs[sid]["warped"]).get_fdata() if sid in regs else None
    _lab_slice, _ = _hemisphere_mask(np.take(lab_vol, sidx, axis=SLICE_AXIS),
                                     meta.get("hemisphere", "whole"))  # FIX: mask labels to kept hemisphere
    anchors = _label_points_on_slice(_lab_slice)
    tmp = tempfile.NamedTemporaryFile(suffix=".svg", delete=False).name
    _graph_to_svg(g, tmp, t1_volume=t1_vol, slice_index=sidx, axis=SLICE_AXIS,
                  label_anchors=anchors, table=table, stroke=stroke,
                  line_mode=meta["line_mode"], double_offset=meta["double_offset"],
                  label_detail=meta["label_detail"],
                  hemisphere=meta.get("hemisphere", "whole"),
                  midline_lr=meta.get("midline_lr"))
    svg_text = open(tmp).read()
    if svg_text.startswith("<?xml"): svg_text = svg_text.split("?>", 1)[1]
    print(f"preview subject={sid} slice={sidx} line_mode={meta['line_mode']} "
          f"label_detail={meta['label_detail']}")
    display(HTML(f'<div style="background:#888;display:inline-block;padding:6px;'
                 f'transform:scale({scale});transform-origin:top left">{svg_text}</div>'))
    return tmp

# NOTE: cell6_export_single_slice() calls read_qa_svg() for its round-trip check, and
# read_qa_svg is defined in CELL 7. Run CELL 7 once, then re-run this cell if you want the
# check; the export itself does not depend on it.
# [runner-controlled] cell6_export_single_slice(stroke=0.36)
# [runner-controlled] preview_qa_slice(stroke=0.4)


In [ ]:
#@ IMPORTABLE
# =====================================================================================
# CELL 7  --  FUSE THE LINES   (arc/node fusion + SATM key points; POLYGONS = THE ATLAS)
# =====================================================================================
# WHAT CHANGED, AND WHY EACH OLD PIECE HAD TO GO
# ----------------------------------------------
#   N_FUSION_SAMPLES = 200
#       A FIXED point count allocates points by ARC LENGTH -- a poor proxy for where the shape
#       needs them. A 2px hairpin and a 300px straight edge both got 200. GONE: the count now
#       comes from the SAGITTA BOUND (AFAM F1a), per anchor-to-anchor segment, taking the max
#       over EVERY subject so a feature only one tracer drew is still sampled densely enough
#       to survive. Straight -> 2 points, exactly.
#
#   graph_to_region_contours() + _order_edge_loop()
#       Rebuilt each region as ONE closed ring by keeping the LARGEST connected component of
#       its boundary. That DROPS ISLANDS and DONUT HOLES outright (measured on a synthetic
#       slice: a region with an island kept 57/77 boundary nodes; the donut kept 30/42). It
#       also averaged every SHARED border TWICE -- once inside region A's loop and once inside
#       region B's -- with different start alignment each time, so the two copies disagreed and
#       left gaps/overlaps between neighbours. GONE.
#
#   _fused_to_region_graph()
#       Wrote every element as (rid, 0), so every region claimed to border background. The
#       second code -- the whole point of arc/node topology -- was thrown away. GONE.
#
#   cf.fuse_slice_contours()
#       Between two nodes it did a pure arc-length pointwise mean. That is exactly the Karcher
#       scheme, the paper's WORST protrusion preserver (AVG_GL -10.450). A notch sitting at 30%
#       of one tracing's arc and 45% of another's got smeared into a bump half its depth.
#       REPLACED by ef.fuse_graphs: SATM's key points (curvature extrema, matched by Hungarian
#       with d_max + zero padding) become INTERIOR anchors, and the average is piecewise
#       BETWEEN them.
#
# WHAT COMES OUT
# --------------
#   fused_graphs  {sidx: (BoundaryGraph, seeds)}   -- both codes intact on every element
#   fused_regions {sidx: {rid: [shapely Polygon]}} -- *** THIS IS THE ATLAS ***
#   fused_anchors {sidx: [(rid, x, y, clearance, n_instances), ...]}
#   fusion_diag   {sidx: {S1..S7 diagnostics}}
#
# Faces get their region id by INTERSECTING THE CODES of the arcs on their boundary (a face
# bounded by (3,5), (3,8), (3,9) belongs to region 3). Whatever that rule cannot decide becomes
# an explicit UNLABELED region (id 8001+, drawn MAGENTA), so a gap left by the QA appears ON
# the atlas instead of silently vanishing into the unbounded face.

# Per-scan input directory holding the QA-edited SVGs exported from Illustrator.
FUSION_INPUT_DIR = f"{RUN_DIR}/qc"      # per-run now; no cross-run bleed to disambiguate
FUSION_GLOB      = "*.svg"
FUSION_INPUT_PATHS = {}          # optional per-scan overrides {sid: full_svg_path}

# Blend weights. None = equal (the unbiased mean). {sid: w} for a weighted mean.
# N-SUBJECT THROUGHOUT (AFAM F7): two subjects with (1-w, w) reproduce the pairwise case;
# SATM's Fig. 4 shows the measures stabilise at n ~ 3.
FUSION_WEIGHTS = None


def _resolve_fusion_inputs(sids, sidx, input_dir=None, input_paths=None):
    """One SVG path per subject. Selection order for each subject's QA SVG:
       1. an explicit override in input_paths[sid]
       2. the RUN_PREFIX path  {input_dir}/{prefix}_{sid}_slice_{sidx:03d}.svg  (if RUN_PREFIX set)
       3. LAST EDITED: newest {sid}_slice_{sidx:03d}.svg in input_dir by mtime, across any prefix.
    """
    input_dir   = input_dir   or FUSION_INPUT_DIR
    input_paths = input_paths if input_paths is not None else FUSION_INPUT_PATHS
    resolved = {}
    for sid in sids:
        if sid in input_paths:                                   # (1) explicit override
            p = input_paths[sid]
        elif os.path.exists(f"{input_dir}/{sid}_slice_{sidx:03d}.svg"):   # (2) this run's file
            p = f"{input_dir}/{sid}_slice_{sidx:03d}.svg"
        else:                                                    # (3) last edited
            cands = glob.glob(f"{input_dir}/*{sid}_slice_{sidx:03d}.svg")
            if not cands:
                raise FileNotFoundError(
                    f"{sid}: no fusion input SVG in {input_dir} for slice {sidx:03d} "
                    f"(run={RUN_PREFIX!r}, dir={input_dir}). Run CELL 6 (auto-QA copies "
                    f"there), or set "
                    f"FUSION_INPUT_PATHS[{sid!r}].")
            p = max(cands, key=os.path.getmtime)
            print(f"  [last-edited] {sid}: no prefix match, using newest -> {p}")  # TROUBLESHOOTING PRINT
        resolved[sid] = p
        print(f"  {sid}: loading {p}")
    return resolved


def _load_fusion_graphs(sidx, input_dir=None, input_paths=None, corrected=None):
    """Get {sid: BoundaryGraph} for one slice. Priority:
       1. `corrected` passed in explicitly (CELL 9's in-memory tests)
       2. the 'corrected' state, if the AVERAGE TEST cell seeded one
       3. the QA-edited SVGs on disk
    The old cell went straight to (3) and blew up in the test flow, because the AVERAGE TEST
    cell writes (2) and nothing read it."""
    if corrected is not None:
        return {sid: sl[sidx] for sid, sl in corrected.items() if sidx in sl}, "argument"
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    H_lr, W_si = np.take(lab_vol, sidx, axis=SLICE_AXIS).shape
    state = load_state("corrected", default=None)
    if state and all(sidx in sl for sl in state.values()):
        return {sid: sl[sidx] for sid, sl in state.items()}, "state 'corrected'"
    sids = list(load_state("propagated"))
    resolved = _resolve_fusion_inputs(sids, sidx, input_dir, input_paths)
    return {sid: read_qa_svg(p, H_lr, W_si) for sid, p in resolved.items()}, "QA SVGs on disk"


def cell7_fuse_lines(input_dir=None, input_paths=None, corrected=None, weights=None,
                     params=None, run_id=None, verbose=True):
    """Fuse the per-subject boundary graphs for the toggled slice, then REBUILD THE POLYGONS."""
    params = params or FUSION_PARAMS
    run_id = FUSION_RUN_ID if run_id is None else run_id
    weights = FUSION_WEIGHTS if weights is None else weights
    meta = load_state("test_meta"); sidx = meta["slice_index"]
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    registry = ef.load_registry(REGISTRY_PATH)

    graphs, src = _load_fusion_graphs(sidx, input_dir, input_paths, corrected)
    if len(graphs) < 1:
        raise ValueError("no subject graphs to fuse")
    print(f"  fusing {len(graphs)} subject(s) from {src}: {list(graphs)}")

    lab_slice = np.take(lab_vol, sidx, axis=SLICE_AXIS)
    lab_slice, _mid = _hemisphere_mask(lab_slice, meta.get("hemisphere", "whole"))
    seeds    = _seed_points_on_slice(lab_slice)
    expected = sorted(set(int(r) for r in np.unique(lab_slice)) - {0})

    fused_graphs, fused_regions, fused_anchors, all_diag = {}, {}, {}, {}
    import time as _time
    stage_times = {}                     # {stage: seconds}, filled by run_fusion (see below)
    _t_wall = _time.perf_counter()
    try:
        # phantom-region notes and Situation-3 hard-stops name the run, slice, and region
        # acronyms; the module has none of these, so hand them in. combined_lut maps region id
        # -> acronym (fallback to the numeric id for anything not in the LUT).
        _acr_map = {int(rid): (combined_lut.get(int(rid), {}) or {}).get("acronym", str(int(rid)))
                    for rid in expected}
        report_ctx = {"run_label": RUN_PREFIX, "slice_index": SLICE_INDEX, "acronyms": _acr_map}
        out = ef.run_fusion(graphs, seeds, expected, weights=weights, params=params,
                            registry=registry, run_id=run_id, report_ctx=report_ctx, slice_index=sidx,
                            verbose=verbose, timings=stage_times)
    except ef.TopologyDivergence as e:
        # The subjects' region-adjacency graphs DISAGREE. Averaging across a topology change
        # produces a locally wrong map with no warning -- SATM degrades gracefully here, the
        # edge/node method has a cliff (SATM C2). Refuse rather than emit a bad atlas.
        print(f"\n*** slice {sidx}: FUSION REFUSED ***\n{e}\n")
        raise

    registry = out["registry"]
    ef.save_registry(registry, REGISTRY_PATH)

    fused_graphs[sidx]  = (out["graph"], seeds)
    fused_node_graphs = {sidx: out["node_graph"]}
    save_state("fused_node_graphs", fused_node_graphs)
    _ng = out["node_graph"]._node_graph_report
    print(f"    node-graph hand-off        : {_ng['n_nodes']} nodes, {_ng['n_elements']} "
          f"elements (every vertex is a node)"
          + ("" if _ng["ok"] else f"  !! dup_segments={_ng['dup_segments']} "
             f"self_loops={_ng['self_loops']}"))
    fused_regions[sidx] = out["regions"]        # {rid: [Polygon]}  <-- THE ATLAS
    fused_anchors[sidx] = out["anchors"]
    all_diag[sidx]      = out["diag"]

    save_state("fused_graphs",  fused_graphs)
    save_state("fused_regions", fused_regions)
    save_state("fused_anchors", fused_anchors)
    save_state("fusion_diag",   all_diag)

    # ---- what the reviewer needs to look at ------------------------------------------
    unl = [f for f in out["flags"] if f[0] == "unlabeled"]
    mrg = [f for f in out["flags"] if f[0] == "merged_regions"]
    d = out["diag"]
    print(f"\n  slice {sidx}: {len(out['arcs'])} fused arcs -> {len(out['faces'])} faces -> "
          f"{len(out['regions'])} regions")
    print(f"    F1  interior anchors placed : {d['S4_kp_anchors_total']} "
          f"(mean {d['S4_kp_anchors_mean']}/arc)")
    print(f"    F1a points allocated        : {d['S4_pts_total']} "
          f"(a fixed N_FUSION_SAMPLES=200 would have written {200*len(out['arcs'])})")
    if "S7_budget_saturated" in d:                                      # only when build_polygons=True
        print(f"    S7_budget_saturated         : {d['S7_budget_saturated']}"
              + ("   <-- a feature your key-point extractor MISSED. Lower kp_alpha, or raise n_max_seg."
                 if d['S7_budget_saturated'] else "   (0 = every segment reached fit_tol)"))
    else:
        print("    S7_budget_saturated         : (skipped -- build_polygons=False)")  # TROUBLESHOOTING PRINT
    print(f"    S3 suggested node_match_max : {d['S3_suggested_node_match_max']} "
          f"(= 3 x measured p95 displacement; you have {params.node_match_max})")
    if unl:
        print(f"\n    !! {len(unl)} UNLABELED region(s) -- a QA gap is now VISIBLE on the atlas:")
        for f in unl:
            print(f"       id {f[1]}  reason={f[2]}  seeds_inside={f[3]}  codes={f[4]}  "
                  f"area={f[5]}px^2")
    if mrg:
        print(f"    !! {len(mrg)} MERGED region pair(s) (a separating boundary is missing): "
              f"{[f[1] for f in mrg]}")

    # ---- FULL-SCALE TIMING (this is your real slice, not a synthetic one) -------------
    # stage_times comes straight from run_fusion. The stages:
    #   fuse             : match nodes + key points, average every arc (F1a sampling)
    #   rebuild          : polygonize the fused arcs into the atlas faces + label them
    #   subject_rebuild  : rebuild EACH subject's own atlas (only if run_comparative_diag)
    #   diag_comparative : the S6/S7 per-region shape metrics (only if run_comparative_diag)
    #   diag_cheap       : S1..S5 pipeline-correctness checks (always; tiny)
    _wall = _time.perf_counter() - _t_wall
    print(f"\n    --- timing (slice {sidx}, {len(graphs)} subjects, {len(out['arcs'])} arcs, "
          f"{len(out['regions'])} regions) ---")
    for _stage in ("fuse", "rebuild", "subject_rebuild", "diag_comparative", "diag_cheap"):
        if _stage in stage_times:
            _sec = stage_times[_stage]
            print(f"      {_stage:18s} {_sec:7.2f}s  ({100*_sec/max(_wall,1e-9):3.0f}%)")
    print(f"      {'TOTAL run_fusion':18s} {_wall:7.2f}s")
    if not params.run_comparative_diag:
        print(f"      (run_comparative_diag=False: S6/S7 + per-subject rebuild skipped. "
              f"Set it True in FUSION_PARAMS for the shape-quality metrics.)")

    report_outputs("CELL 7 fuse (arc/node + SATM key points)",
                   files=[REGISTRY_PATH],
                   state_keys=["fused_graphs", "fused_regions", "fused_anchors", "fusion_diag"],
                   checks=[("no UNLABELED faces (no QA gaps)", not unl,
                            "look at the magenta regions in CELL 8's render"),
                           ("no merged regions", not mrg, "a separating boundary is missing"),
                           ("polygonize did not fall back", not d.get("S5_fallback_chainer", False),
                            f"polygonize error: {d.get('S5_polygonize_error')}"),
                           ("Euler identity holds (V-E+F == C)", d.get("G_euler_defect", 0) == 0,
                            "an arc, node or face was lost or duplicated")])
    return fused_graphs

# [runner-controlled] cell7_fuse_lines()


In [ ]:
#@ IMPORTABLE
# =====================================================================================
# CELL 8  --  RENDER THE FUSED ATLAS   (lines + the POLYGON atlas + diagnostics)
# =====================================================================================
# Two things are rendered, and the second one is the deliverable:
#   * the fused LINES, over the T1 underlay -- with the per-subject inputs faint behind them,
#     so a correct average visibly sits in the MIDDLE of its inputs;
#   * the fused POLYGONS, filled. THE POLYGONS ARE THE ATLAS (this is what the volumetric
#     conversion consumes). Any MAGENTA face is an UNLABELED region: a gap the QA left, now
#     visible instead of silently missing.
#
# WHAT IS **NOT** DONE HERE, AND WHY (C13)
# ----------------------------------------
# The export IS the direct output of the fusion. No re-smoothing, no re-resampling. The point
# count was already set by F1a's sagitta bound, per anchor-to-anchor segment, verified against
# the actual chord deviation. Smoothing it again (the old SMOOTH_SIGMA = 3.0) would undo
# exactly the notch preservation the key-point machinery just paid for -- and it would do so
# AFTER the metrics said it worked.
#
# The old cell also referenced `lab_vol` and `sidx` at module scope inside its
# `except FileNotFoundError:` fallback, before either name existed. That was a latent NameError
# on any run that did not take the CELL-5-seeded path. Gone.

OVERLAY_INPUTS = True     # draw the per-subject inputs (gray) behind the fused line
FUSED_STROKE   = 0.36
ATLAS_FILL     = 0.55     # fill opacity for the polygon atlas. 0 = lines only.


def _fused_underlay_volume():
    """Any subject's warped template-space image works (they share the template grid)."""
    regs = load_state("regs", default={})
    for sid, r in (regs or {}).items():
        if r.get("warped") and os.path.exists(r["warped"]):
            return nib.load(r["warped"]).get_fdata()
    return nib.load(TEMPLATE_T1).get_fdata()


def render_fused_slice_svg(overlay_inputs=OVERLAY_INPUTS, stroke=FUSED_STROKE,
                           atlas_fill=ATLAS_FILL, scale=2.0):
    from IPython.display import HTML, display
    meta   = load_state("test_meta"); sidx = meta["slice_index"]
    fused  = load_state("fused_graphs")
    if sidx not in fused:
        raise KeyError(f"no fused graph for slice {sidx}; run CELL 7 first.")
    g_fused = fused[sidx][0]
    # LINES-ONLY MODE (build_polygons=False): fused_regions/anchors are empty. Draw lines only.
    regions = load_state("fused_regions").get(sidx, {})
    anchors = load_state("fused_anchors").get(sidx, [])
    # Labels come from fused_anchors ONLY when polygons were built. With build_polygons=False
    # that list is empty, which is why the fused render lost its labels. Fall back to the SAME
    # label-anchor rule the QA export uses (_label_points_on_slice: one anchor per region
    # component at the distance-transform maximum), computed from the template label slice masked
    # to the kept hemisphere. This places a label at the deepest interior point of every region,
    # so it tracks the region as its shape moves. No polygon rebuild required.

    # BASED ON QA regions, NOT FUSED regions. They will be slightly off (unless polygons are activated)
    if not anchors:
        _lab_vol_lbl = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
        _lab_slice_lbl, _ = _hemisphere_mask(np.take(_lab_vol_lbl, sidx, axis=SLICE_AXIS),
                                             meta.get("hemisphere", "whole"))
        anchors = _label_points_on_slice(_lab_slice_lbl)
        print(f"  [labels] fused_anchors empty (build_polygons=False) -> "  # TROUBLESHOOTING PRINT
              f"placed {len(anchors)} label anchor(s) from the template label slice.")

    table   = load_region_table(TEMPLATE_LUT)
    t1_vol  = _fused_underlay_volume()

    out_svg = f"{RUN_DIR}/atlas/fused_slice_{sidx:03d}.svg"
    _graph_to_svg(g_fused, out_svg, t1_volume=t1_vol, slice_index=sidx, axis=SLICE_AXIS,
                  label_anchors=anchors, table=table, stroke=stroke,
                  line_mode="single",                 # the fused atlas is single-line by
                  double_offset=meta["double_offset"],#   construction: each shared border is
                  label_detail=meta["label_detail"],  #   stored and averaged EXACTLY ONCE
                  hemisphere=meta.get("hemisphere", "whole"),
                  midline_lr=meta.get("midline_lr"),
                  smooth_sigma=0.0,                   # C13: the export IS the fusion output
                  regions=regions, fill_opacity=atlas_fill)
    report_outputs("CELL 8 render fused atlas", files=[out_svg])

    svg_text = open(out_svg).read()
    if svg_text.startswith("<?xml"): svg_text = svg_text.split("?>", 1)[1]

    if overlay_inputs:
        graphs, src = _load_fusion_graphs(sidx)
        png, W, H, (H_lr, W_si) = _slice_to_png_bytes(t1_vol, sidx, SLICE_AXIS)
        overlay = [f'<g fill="none" stroke="#222" stroke-width="{stroke*0.6:.3f}" opacity="0.5">']
        for sid, gi in graphs.items():
            for arc in ef.graph_to_arcs(gi):
                P = np.vstack([arc.pts, arc.pts[0]]) if arc.closed else arc.pts
                D = _orient_arr(P, H_lr, W_si)
                if len(D) < 2: continue
                overlay.append('<polyline points="%s"/>' %
                               " ".join(f"{x:.2f},{y:.2f}" for x, y in D))
        overlay.append('</g>')
        gt = svg_text.find(">")
        svg_text = svg_text[:gt+1] + "\n" + "\n".join(overlay) + svg_text[gt+1:]

    unl = sorted(r for r in regions
                 if FUSION_PARAMS.unlabeled_id_start <= r < FUSION_PARAMS.new_id_start)
    new = sorted(r for r in regions if r >= FUSION_PARAMS.new_id_start)
    print(f"fused atlas: slice={sidx}  {len(regions)} regions, "
          f"{sum(len(f) for f in regions.values())} faces  (fill={atlas_fill})")
    print(f"  bold color = the fused line;  dark thin = the per-subject inputs")
    if unl: print(f"  !! MAGENTA regions {unl} are UNLABELED -- QA gaps, visible by design")
    if new: print(f"  expert-drawn NEW regions: {new}")
    display(HTML(f'<div style="background:#888;display:inline-block;padding:6px;'
                 f'transform:scale({scale});transform-origin:top left">{svg_text}</div>'))
    return out_svg


def print_fusion_report(verbose=False):
    """The S1..S7 table for the toggled slice. S7 is the part that tells you whether the fused
    SHAPE is a good shape -- S1..S6 only tell you the PIPELINE ran correctly, and the paper's
    central finding is that a method can score the BEST Jaccard while producing the WORST
    HD, PERI, TOPO and AVG_GL (GEMS buys overlap by breaking topology)."""
    meta = load_state("test_meta"); sidx = meta["slice_index"]
    d = load_state("fusion_diag")[sidx]
    return ef.print_report(d, f"slice {sidx}", verbose=verbose)


def print_fusion_params(p=None):
    """Echo the exact FusionParams this fused output was produced with, so the render and the
    numbers above it are self-documenting. See the parameter reference doc for what each means."""
    import dataclasses as _dc
    p = p or FUSION_PARAMS
    print("\n=== FUSION PARAMETERS used for this fused output ===")
    print(f"  RUN_PREFIX = {RUN_PREFIX!r}   FUSION_RUN_ID = {FUSION_RUN_ID}   "
          f"weights = {FUSION_WEIGHTS or 'equal'}")
    print(f"  RUN_DIR    = {RUN_DIR}")
    for f in _dc.fields(p):
        print(f"    {f.name:22s} = {getattr(p, f.name)!r}")
    print("=" * 52)

#render_fused_slice_svg()
#print_fusion_report()
#print_fusion_params()


## 2 · Orchestration machinery *(rarely edited)*

Timing + result capture, the stage model, the parcellation cache, the metric key and the append-only store, the perturbation field, the SVG dumps, and the stage runner.

In [ ]:

#@ IMPORTABLE
# =====================================================================================
# ORCHESTRATION MACHINERY   (the experiment layer; TEST v19 is untouched)
# =====================================================================================
import os, glob, time, pickle, itertools, io, contextlib, hashlib, json, math, traceback
import dataclasses as _dc
import numpy as np
from scipy.spatial import cKDTree

# ---- (v3 FIX A) capture the timings AND the return value of ef.run_fusion -------------
# v2 wrapped run_fusion only for `timings`. The RETURN dict also carries
# result.stats["arcs"] (per-arc saturation / fit residual / point count), which lets the
# FUSE stage report S7_budget_saturated and S7_fit_residual_px WITHOUT polygonizing.
# Still zero edits to TEST.
LAST_TIMINGS, LAST_FUSION = {}, {}
if not getattr(ef.run_fusion, "_v3_wrapped", False):
    _orig_run_fusion = ef.run_fusion
    def _wrapped_run_fusion(*a, **kw):
        t = kw.get("timings")
        if t is None:
            t = {}
            kw["timings"] = t
        LAST_TIMINGS.clear(); LAST_FUSION.clear()
        out = _orig_run_fusion(*a, **kw)
        LAST_TIMINGS.update(t)
        LAST_FUSION.update(out)
        return out
    _wrapped_run_fusion._v3_wrapped = True
    _wrapped_run_fusion._timing_wrapped = True
    ef.run_fusion = _wrapped_run_fusion
    print("[wrap] ef.run_fusion -> LAST_TIMINGS + LAST_FUSION")   # TROUBLESHOOTING PRINT


class Sections:
    """Wall-clock per SECTION of this notebook (parcellation, perturb, fuse, dumps).
    Fusion's INTERNAL stages come from LAST_TIMINGS; both end up in the results row."""
    def __init__(self): self.t = {}
    def __call__(self, name):
        self._n = name; return self
    def __enter__(self): self._t0 = time.perf_counter(); return self
    def __exit__(self, *e): self.t[self._n] = round(time.perf_counter() - self._t0, 3)


def canonical_name(i): return f"subject{i+1}"

DUMP_SVG = dict(after_perturb=False, after_fuse=False, after_polygonize=False,
                perturb_overlay=False, fused_overlay=False)   # <<< EDIT: overlay dumps

# ---- STAGES --------------------------------------------------------------------------
# The pipeline is a CHAIN; a stage cannot start before the one above it produced its
# output. "Running a stage independently" therefore means: RESTORE the earlier stages
# from the parcellation cache, RUN this one, STOP.
#
# fuse and poly are ONE call to ef.run_fusion -- polygonize CONSUMES the fused arcs, so
# "poly without fuse" does not exist. `poly` means fuse+polygonize. Consequences, read
# straight off run_fusion():
#     S1..S4  always produced          -> available at `fuse`
#     S5, G_* need build_polygons      -> `poly`
#     S6, S7  need build_polygons AND run_comparative_diag -> `poly`
# The two exceptions are S7_budget_saturated and S7_fit_residual_px, which are computed
# inside diag_shape but derive ONLY from result.stats["arcs"]; the runner recomputes them
# at the fuse stage from LAST_FUSION.
STAGE_ORDER = ("parc", "pert", "fuse", "poly")

def stages_upto(stages):
    """Normalise a stage list to a PREFIX of STAGE_ORDER, so a plan can never ask for a
    later stage without the earlier ones."""
    s = set(stages)
    last = max((i for i, n in enumerate(STAGE_ORDER) if n in s), default=0)
    return STAGE_ORDER[:last + 1]

# ---- parcellation cache --------------------------------------------------------------
CACHE_ROOT = f"{WORK_TEST}/_postqa_cache"
PARCELLATION_KEYS = {"slice_index", "hemisphere", "smooth_sigma", "smooth_boundaries",
                     "simplify_tol", "trace_max_handles", "line_mode"}

def _hash(d):
    return hashlib.md5(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:10]

def cache_key(cfg):
    parc = {k: cfg.get(k) for k in sorted(PARCELLATION_KEYS)}
    return f"sl{parc.get('slice_index')}_{parc.get('hemisphere')}_{_hash(parc)}"

def cache_dir_for(cfg): return f"{CACHE_ROOT}/{cache_key(cfg)}"
def cache_exists(cfg):  return os.path.exists(f"{cache_dir_for(cfg)}/postload.pkl")

def needs_full_stack(cfg, swept):
    if PARCELLATION_KEYS & set(swept): return True, "a parcellation parameter is being swept"
    if not cache_exists(cfg):          return True, "no post-load cache exists for this key yet"
    return False, "cache hit -> parcellation skipped"

# ---- (v3 FIX B) per-run REGISTRY_PATH ------------------------------------------------
# CELL A computed REGISTRY_PATH ONCE from the FIRST RUN_DIR, so every run in a batch read
# and wrote the SAME region_registry.json. registry_next_new / registry_next_unlabeled
# allocate 8001+/9001+ ids MONOTONICALLY from that file, so run 7's "new region" ids
# depended on runs 1-6 and no two runs were comparable. Each run now gets its own.
def set_run_paths(run_dir):
    g = globals()
    g["RUN_DIR"] = run_dir
    g["REGISTRY_PATH"] = f"{run_dir}/atlas/region_registry.json"
    g["FUSION_INPUT_DIR"] = f"{run_dir}/qc"                     # folders are made by their writers


os.makedirs(CACHE_ROOT, exist_ok=True)
print(f"[stages] {STAGE_ORDER}   cache -> {CACHE_ROOT}")


In [ ]:

#@ IMPORTABLE
# =====================================================================================
# METRIC KEY + APPEND-ONLY RESULTS STORE            (this is the source of truth)
# =====================================================================================
# STORAGE MODEL, and why it is this one:
#   runs.jsonl   APPEND-ONLY, one JSON object per finished run. Source of truth. A crash
#                can only truncate the LAST line, and the reader skips a bad last line,
#                so a completed run is never lost. It DOUBLES AS THE RESUME REGISTRY,
#                so there is no second file that can fall out of sync with it.
#   Everything else (wide CSV, long CSV, summary CSV, metrics.xlsx, plots) is DERIVED
#   and can be deleted and rebuilt from runs.jsonl at any time. Nothing is ever edited
#   in place, so a spreadsheet cannot desynchronise from the data.
RESULTS_DIR    = f"{WORK_TEST}/_results"
RUNS_JSONL     = f"{RESULTS_DIR}/runs.jsonl"
METRIC_KEY_CSV = f"{WORK_TEST}/METRIC_KEY.csv"     # the PARENT dir, as requested
for _d in (RESULTS_DIR, f"{RESULTS_DIR}/plots"):
    os.makedirs(_d, exist_ok=True)

METRIC_KEY = {
 # ---- P: parcellation stage ----------------------------------------------------------
 "P_n_regions":"P_nr", "P_n_arcs":"P_na", "P_n_nodes":"P_nn", "P_n_subjects":"P_ns",
 "P_pts_total":"P_pt",
 # ---- X: perturbation stage ----------------------------------------------------------
 "X_amp_req":"X_ar", "X_lam_req":"X_lr", "X_amp_ach_max":"X_axm", "X_amp_ach_rms":"X_axr",
 "X_sign_sum":"X_ss", "X_arcs_dropped":"X_ad", "X_drop_frac_ach":"X_dfa", "X_seeds":"X_sd",
 "X_tdev_max_px":"X_tdm", "X_tdev_p95_px":"X_tdp",
 # ---- S1 parse -----------------------------------------------------------------------
 "S1_label_yield":"S1_ly", "S1_curve_dev_px":"S1_cd", "S1_selfpair_count":"S1_sp",
 "S1_code_degree_min":"S1_cdm", "S1_n_arcs":"S1_na",
 # ---- S2 nodes -----------------------------------------------------------------------
 "S2_n_nodes":"S2_nn", "S2_dangling_ends":"S2_de", "S2_degree_hist":"S2_dh",
 "S2_intra_over_inter":"S2_ioi", "S2_min_inter_node_gap_px":"S2_mig",
 "S2_short_arcs":"S2_sa", "S2_short_arc_codes":"S2_sac", "S2_dangling_coords":"S2_dc",
 # ---- S3 node match ------------------------------------------------------------------
 "S3_fused_nodes":"S3_fn", "S3_node_match_rate":"S3_nmr", "S3_node_disp_p95_px":"S3_ndp",
 "S3_node_disp_max_px":"S3_ndm", "S3_suggested_node_match_max":"S3_snm",
 "S3_node_rejected_far":"S3_nrf", "S3_regionset_collisions":"S3_rc",
 "S3_collision_sets":"S3_cs", "S3_rejected_detail":"S3_rd", "S3_unmatched":"S3_um",
 # ---- S4 arc fusion ------------------------------------------------------------------
 "S4_n_fused_arcs":"S4_nfa", "S4_snap_dist_max_px":"S4_sdm", "S4_arcs_dropped":"S4_ad",
 "S4_arcs_partial":"S4_ap", "S4_dir_ambiguous":"S4_da", "S4_support_hist":"S4_sh",
 "S4_kp_anchors_total":"S4_kat", "S4_kp_anchors_mean":"S4_kam",
 "S4_kp_nonmonotone":"S4_knm", "S4_orphan_policy":"S4_op", "S4_pts_total":"S4_pt",
 # ---- S5 rebuild / polygonize --------------------------------------------------------
 "S5_n_faces":"S5_nf", "S5_region_yield":"S5_ry", "S5_regions_missing":"S5_rm",
 "S5_regions_extra":"S5_re", "S5_unlabeled_faces":"S5_uf", "S5_unlabeled_detail":"S5_ud",
 "S5_orphan_arcs":"S5_oa", "S5_orphan_codes":"S5_oc", "S5_merged_regions":"S5_mr",
 "S5_arc_crossings":"S5_ac", "S5_rings_nonsimple":"S5_rn", "S5_nonsimple_ids":"S5_ni",
 "S5_dangling_repaired":"S5_dr", "S5_dangling_unrepaired":"S5_du",
 "S5_dropped_degenerate":"S5_dd", "S5_dropped_duplicate":"S5_dpd",
 "S5_face_code_conflicts":"S5_fcc", "S5_fallback_chainer":"S5_fc",
 "S5_polygonize_error":"S5_pe", "S5_sliver_faces":"S5_sf", "S5_topo_delta_regions":"S5_tdr",
 # ---- G: planar-graph identity -------------------------------------------------------
 "G_V":"G_V", "G_E":"G_E", "G_F":"G_F", "G_components":"G_C",
 "G_euler_defect":"G_ed", "G_overlap_area":"G_oa",
 # ---- S6 overlap ---------------------------------------------------------------------
 "S6_iou_area_weighted":"S6_iaw", "S6_iou_mean_per_subject":"S6_imps",
 "S6_iou_worst_region":"S6_iwr", "S6_hausdorff_worst_region":"S6_hwr",
 "S6_area_cons_dev":"S6_acd",
 # ---- S7 shape -----------------------------------------------------------------------
 "S7_budget_saturated":"S7_bs", "S7_fit_residual_px":"S7_fr", "S7_representation":"S7_rep",
 "S7_peri_dev_mean_signed":"S7_pdms", "S7_peri_dev_max":"S7_pdx",
 "S7_round_dev_max":"S7_rdm", "S7_round_dev_worst_region":"S7_rdwr",
 "S7_avg_gl_dev_mean":"S7_agm", "S7_avg_gl_dev_worst_region":"S7_agwr",
 "S7_topo_delta":"S7_td", "S7_topo_delta_regions":"S7_tdr",
 "S7_hd_asym":"S7_ha", "S7_hd_asym_region":"S7_har",
 "S7_curv_ks":"S7_cks", "S7_curv_ks_worst_region":"S7_ckwr", "S7_skele_dev":"S7_sk",
 "S7_sweep_identity_px":"S7_sip", "S7_sweep_identity_bound_px":"S7_sib",
 "S7_sweep_identity_ok":"S7_sio", "S7_sweep_identity_per_subject":"S7_sips",
 # ---- T: timings ---------------------------------------------------------------------
 "t_parcellation":"T_parc", "t_cache_load":"T_cload", "t_duplicate":"T_dup",
 "t_drop":"T_drop", "t_perturb":"T_pert", "t_fuse":"T_fuse", "t_tdev":"T_tdev",
 "t_dump_perturb":"T_dpert", "t_dump_fuse":"T_dfuse", "t_dump_polygonize":"T_dpoly",
 "t_total":"T_tot",
 "fuse_fuse":"T_ffuse", "fuse_rebuild":"T_frb", "fuse_subject_rebuild":"T_fsrb",
 "fuse_diag_cheap":"T_fdc", "fuse_diag_comparative":"T_fdcmp",
}
assert len(set(METRIC_KEY.values())) == len(METRIC_KEY), "metric shorthand collision"

_FAM_STAGE = {"P":"parc", "X":"pert", "S1":"fuse", "S2":"fuse", "S3":"fuse", "S4":"fuse",
              "S5":"poly", "G":"poly", "S6":"poly", "S7":"poly", "t":"-", "fuse":"-"}
_LOWER_BETTER = ("dev", "disp", "resid", "defect", "dropped", "unrepaired", "collis",
                 "rejected", "merged", "unlabeled", "crossings", "nonsimple", "orphan",
                 "saturated", "asym", "_ks", "delta", "sliver", "conflict", "error")
_HIGHER_BETTER = ("yield", "match_rate", "iou", "_ok")

def _family(full):
    for f in ("S1", "S2", "S3", "S4", "S5", "S6", "S7", "G", "P", "X"):
        if full.startswith(f + "_"): return f
    return "fuse" if full.startswith("fuse_") else "t"

def write_metric_key(path=None):
    """The KEY the shorthand refers to. Written to WORK_TEST (the parent), not per-run.
    Hard thresholds come straight from ef.THRESHOLDS, so the key cannot drift from code."""
    import csv as _csv
    path = path or METRIC_KEY_CSV
    thr = getattr(ef, "THRESHOLDS", {})
    with open(path, "w", newline="") as f:
        w = _csv.writer(f)
        w.writerow(["short", "full", "family", "first_stage", "better", "hard_threshold"])
        for full, short in METRIC_KEY.items():
            fam = _family(full)
            better = ("higher" if any(t in full for t in _HIGHER_BETTER)
                      else "lower" if any(t in full for t in _LOWER_BETTER) else "n/a")
            t = thr.get(full)
            w.writerow([short, full, fam, _FAM_STAGE.get(fam, "-"), better,
                        f"{t[0]} {t[1]}" if t else ""])
    print(f"[key] {len(METRIC_KEY)} metrics -> {path}")   # TROUBLESHOOTING PRINT
    return path

def _jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    if isinstance(o, dict):           return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):  return [_jsonable(v) for v in o]
    return o

def append_run(rec, path=None):
    with open(path or RUNS_JSONL, "a") as f:
        f.write(json.dumps(_jsonable(rec), default=str) + "\n")

def read_runs(path=None):
    p = path or RUNS_JSONL
    out = []
    if not p or not os.path.exists(p): return out
    with open(p) as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            try:
                out.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"  [store] skipping malformed line {ln} "      # TROUBLESHOOTING PRINT
                      f"(a crash truncated it; that run will be redone)")
    return out

def done_prefixes(ok_only=True, path=None):
    """RESUME REGISTRY. A prefix that already finished is skipped on the next pass.
    A run with a DIFFERENT prefix always goes through."""
    return {r["prefix"] for r in read_runs(path) if r.get("ok") or not ok_only}

write_metric_key()


In [ ]:

#@ IMPORTABLE
# =====================================================================================
# LOAD -> RENAME/DETACH -> (DUPLICATE) -> (DROP ARCS) -> PERTURB   (the pre-fusion stage)
# =====================================================================================
from collections import defaultdict as _dd

def _code_of(e):
    a, b = int(e[2]), int(e[3])
    return (a, b) if a <= b else (b, a)

def element_arcs(g):
    """Same maximal-chain grouping as ef.graph_to_arcs, but returns ELEMENT INDICES:
    [(code, closed, [element indices...]), ...].

    WHY THIS EXISTS: BoundaryGraph.elements are single SEGMENTS [i, j, matA, matB]; there
    is no 'arc' object to delete. v2's drop_arcs deleted a fraction of SEGMENTS, which
    punches holes in every arc instead of removing whole arcs. Measured on a 40-arc /
    2000-segment graph: frac=0.05 turned 40 arcs into 129 FRAGMENTS, frac=0.15 into 294.
    orphan_policy then saw hundreds of shredded stubs rather than a few missing arcs, so
    E6 never tested orphan policy at all.
    Verified against ef.graph_to_arcs on a hand-built graph (two junction-split arcs plus
    one island loop): identical arc count, codes and closed flags, and an exact partition
    of the element list."""
    inc = _dd(list)
    for ei, e in enumerate(g.elements):
        inc[int(e[0])].append(ei); inc[int(e[1])].append(ei)

    def is_junction(n):
        eis = inc[n]
        if len(eis) != 2: return True
        return _code_of(g.elements[eis[0]]) != _code_of(g.elements[eis[1]])

    junc = {n for n in inc if is_junction(n)}
    used, out = set(), []

    def walk(start_node, start_e, stop_at_junction):
        eis, cur_n, cur_e = [], start_node, start_e
        while True:
            used.add(cur_e); eis.append(cur_e)
            e = g.elements[cur_e]
            nxt = int(e[1]) if int(e[0]) == cur_n else int(e[0])
            if (stop_at_junction and nxt in junc) or nxt == start_node:
                return eis, nxt
            cand = [k for k in inc[nxt] if k != cur_e and k not in used]
            if not cand: return eis, nxt
            cur_e, cur_n = cand[0], nxt

    for j in sorted(junc):
        for ei in list(inc[j]):
            if ei in used: continue
            eis, end = walk(j, ei, True)
            out.append((_code_of(g.elements[ei]), end == j, eis))
    for ei, e in enumerate(g.elements):
        if ei in used: continue
        eis, end = walk(int(e[0]), ei, False)
        out.append((_code_of(e), end == int(e[0]), eis))
    return out


def _normals_at(P, closed=False):
    """Unit normals from a CENTRAL-DIFFERENCE tangent, so a vertex's normal does not
    depend on which direction its arc happens to be traversed."""
    P = np.asarray(P, float)
    if len(P) < 2: return np.zeros_like(P)
    T = (np.roll(P, -1, 0) - np.roll(P, 1, 0)) / 2.0 if closed else np.gradient(P, axis=0)
    L = np.hypot(T[:, 0], T[:, 1]); L[L < 1e-12] = 1.0
    T = T / L[:, None]
    return np.c_[-T[:, 1], T[:, 0]]


def _field_2d(P, amp, lam_px, rng, n_waves=6):
    """Isotropic band-limited VECTOR displacement at 2-D positions P, wavelength lam_px in
    PIXELS. Each component is a sum of n_waves plane waves (random direction + phase); the
    pair is normalised together so max|D| == amp exactly.

    VECTOR, not scalar-along-normal: the old version took its normal from the ORDER of the
    .nodes array, which is along-arc order only WITHIN one traced boundary and arbitrary at
    every boundary seam and reused junction. A vector field has no such dependency, and it
    is what synthetic registration warps are in the literature.

    amp == 0 returns EXACT zeros -- that is what makes E0.3 a bit-identity test."""
    if amp == 0.0 or lam_px <= 0:
        return np.zeros((len(P), 2))
    def _component():
        d = np.zeros(len(P))
        for _ in range(n_waves):
            th = rng.uniform(0, 2 * np.pi)
            u = np.array([np.cos(th), np.sin(th)])
            d += np.sin(2 * np.pi * (P @ u) / lam_px + rng.uniform(0, 2 * np.pi))
        return d
    D = np.column_stack([_component(), _component()])
    m = float(np.hypot(D[:, 0], D[:, 1]).max())
    return amp * D / m if m > 1e-12 else D


def load_rename_detach():
    prop = load_state("propagated")
    meta = load_state("test_meta"); sidx = meta["slice_index"]
    ren, name_map = {}, {}
    for i, sid in enumerate(prop):
        ren[canonical_name(i)] = bg.read_ascii(prop[sid][sidx])
        name_map[canonical_name(i)] = sid
    return ren, name_map, sidx


def duplicate_to(ren, name_map, n_target):
    """Pad to n_target by copying the (identical pre-perturbation) map.
    (v3 FIX C) v2 only ever GREW the set: with 3 real subjects and n_subjects=2 you
    silently fused 3, and E5's N axis quietly stopped being an axis."""
    ren, name_map = dict(ren), dict(name_map)
    reals = list(ren)
    if n_target < len(ren):
        keep = reals[:n_target]
        print(f"  n_subjects={n_target} < {len(ren)} loaded -> keeping {keep}")  # TROUBLESHOOTING PRINT
        return {k: ren[k] for k in keep}, {k: name_map[k] for k in keep}
    i = len(ren)
    while len(ren) < n_target:
        src = reals[i % len(reals)]
        ren[canonical_name(i)] = bg.BoundaryGraph(
            nodes=np.array(ren[src].nodes).copy(), elements=[e[:] for e in ren[src].elements])
        name_map[canonical_name(i)] = name_map[src]
        i += 1
    return ren, name_map


def drop_arcs(ren, frac, seed):
    """(v3 FIX D) Delete a fraction of WHOLE ARCS per subject, independently seeded.

    An orphan is an arc group with len(group) < n_subjects. Every subject starts as an
    IDENTICAL copy of the template and the displacement field only MOVES nodes, so every
    group otherwise contains all N subjects and the orphan branch never executes. This is
    the ONLY thing that makes orphan_policy / min_arc_support testable (E6)."""
    if frac <= 0:
        return ren, {"X_arcs_dropped": 0, "X_drop_frac_ach": 0.0}
    out, tot_arcs, drop_arcs_n = {}, 0, 0
    for j, (nm, g) in enumerate(ren.items()):
        rng = np.random.default_rng(seed + 104729 * j)          # independent per subject
        groups = element_arcs(g)
        kill = set()
        n_killed = 0
        for _code, _closed, eis in groups:
            if rng.random() < frac:
                kill.update(eis); n_killed += 1
        keep = [e for i, e in enumerate(g.elements) if i not in kill]
        tot_arcs += len(groups); drop_arcs_n += n_killed
        out[nm] = bg.BoundaryGraph(nodes=np.array(g.nodes).copy(),
                                   elements=[e[:] for e in keep])
        print(f"  {nm}: dropped {n_killed}/{len(groups)} WHOLE arcs "
              f"({len(g.elements)-len(keep)}/{len(g.elements)} elements)")
    return out, {"X_arcs_dropped": drop_arcs_n,
                 "X_drop_frac_ach": round(drop_arcs_n / max(tot_arcs, 1), 4)}


def perturb_all(ren, amp, lam_px, mode, seed, sidx, n_waves=6):
    """'anti' needs signs that SUM TO ZERO so the fused result must return to the
    unperturbed template -- that is the ground-truth case. linspace(+1,-1,N) gives the
    MIDDLE subject a sign of exactly 0.0 at odd N, silently leaving one of three
    'independent' subjects unperturbed, so the ladder is used for 'anti' ONLY. 'same' and
    'indep' use all-ones and differ only in whether the SEED advances per subject."""
    names = list(ren)
    if len(names) == 1:
        signs = [0.0]
    elif mode == "anti":
        signs = list(np.linspace(+1.0, -1.0, len(names)))       # sums to 0 by construction
    else:
        signs = [1.0] * len(names)
    corrected, seeds_used, mx, sq, n = {}, [], 0.0, 0.0, 0
    for j, nm in enumerate(names):
        g = bg.BoundaryGraph(nodes=np.array(ren[nm].nodes).copy(),
                             elements=[e[:] for e in ren[nm].elements])
        nodes = np.asarray(g.nodes, float).copy()
        seed_j = int(seed + (j * 7919 if mode == "indep" else 0))   # only indep varies
        rng = np.random.default_rng(seed_j)
        D = _field_2d(nodes, amp, lam_px, rng, n_waves) * signs[j]
        g.nodes = nodes + D
        d = np.hypot(D[:, 0], D[:, 1])          # magnitudes; the stats below are unchanged
        corrected[nm] = {sidx: g}
        seeds_used.append(seed_j)
        mx = max(mx, float(np.abs(d).max()) if len(d) else 0.0)
        sq += float(np.sum(d ** 2)); n += len(d)
        print(f"  {nm}: sign {signs[j]:+.2f}  max|disp|={np.abs(d).max():.3f} vox  seed={seed_j}")
    _strain = 2 * math.pi * amp / max(lam_px, 1e-9)     # >1 => the warp can fold on itself
    if _strain > 1.0:
        print(f"  !! strain 2pi*A/lam = {_strain:.2f} > 1: this field FOLDS; boundaries may self-intersect")
    stats = {"X_amp_req": float(amp), "X_lam_req": float(lam_px),
             "X_amp_ach_max": round(mx, 4),
             "X_amp_ach_rms": round(math.sqrt(sq / max(n, 1)), 4),
             "X_strain": round(_strain, 3),
             "X_sign_sum": round(float(sum(signs)), 6), "X_seeds": seeds_used}
    if mode == "anti":
        print(f"  mode=anti: signs sum to {sum(signs):+.3f} -> fused must return to template")
    return corrected, signs, stats


# ---- deviation of the fused lines from the UNPERTURBED template ----------------------
# WHY: mode='anti' has a KNOWN ground truth (signs sum to zero, so a correct fusion must
# come back to the template) but v2 never measured against it -- every E3/E4 number was a
# self-consistency number. X_tdev IS the error, and it needs NO polygons, which is what
# makes a cheap FUSE-only screening pass scientifically meaningful rather than just cheap.
#
# LIMIT: a symmetric nearest-neighbour distance between two POINT SETS sampled every
# TDEV_STEP px, so it has a floor of ~TDEV_STEP/2 (measured 0.1256 px at step 0.25 against
# a theoretical 0.125). Read X_tdev_p95 as the HEADLINE and X_tdev_max as a LOCATOR: max
# is dominated by sharp corners, exactly like S6_*_worst_region.
TDEV_STEP = 0.25

def _densify(P, step):
    P = np.asarray(P, float)
    if len(P) < 2: return P
    seg = np.hypot(*np.diff(P, axis=0).T)
    L = np.concatenate([[0.0], np.cumsum(seg)])
    if L[-1] <= 0: return P[:1]
    t = np.linspace(0.0, L[-1], max(int(np.ceil(L[-1] / step)) + 1, 2))   # ENDPOINT INCLUDED
    return np.c_[np.interp(t, L, P[:, 0]), np.interp(t, L, P[:, 1])]

def _arc_cloud(arcs, step):
    pts = []
    for a in arcs:
        P = np.vstack([a.pts, a.pts[:1]]) if a.closed else a.pts
        pts.append(_densify(P, step))
    return np.vstack(pts) if pts else np.zeros((0, 2))

def curve_dev(arcs_a, arcs_b, step=TDEV_STEP):
    A, B = _arc_cloud(arcs_a, step), _arc_cloud(arcs_b, step)
    if not len(A) or not len(B):
        return {"X_tdev_max_px": None, "X_tdev_p95_px": None}
    d = np.concatenate([cKDTree(B).query(A)[0], cKDTree(A).query(B)[0]])
    return {"X_tdev_max_px": round(float(d.max()), 4),
            "X_tdev_p95_px": round(float(np.percentile(d, 95)), 4)}


In [ ]:
#@ IMPORTABLE

# =====================================================================================
# FORCED-SVG DUMPS   (each honours one DUMP_SVG toggle; every skip prints its reason)
# =====================================================================================
# (FIX 4) cell7_fuse_lines RETURNS fused_graphs = {sidx: (BoundaryGraph, seeds)} -- not a dict
# with "graph"/"regions" keys. The fused geometry is read from the SAVED STATES instead, which
# is also what CELL 8 does, so the dumps see exactly what the real notebook sees.
# EVERY dump writes a FILE to a directory under RUN_DIR and prints its absolute path.

def _bg_volume_for(name_map, nm):
    """Underlay for one subject, template as fallback: _graph_to_svg writes an EMPTY file
    when t1_volume is None, so duplicates and the fused map need a volume too."""
    regs = load_state("regs", default={})
    real = name_map.get(nm)
    if real and real in regs and "warped" in regs[real]:
        try:    return nib.load(regs[real]["warped"]).get_fdata()
        except Exception as e:
            print(f"    [dump] background load failed for {nm} <- {real}: {e}")
    return nib.load(TEMPLATE_T1).get_fdata()

def dump_graph_svg(graph, sidx, name_map, nm, tag, subdir, stroke=0.36):
    meta  = load_state("test_meta")
    table = load_region_table(TEMPLATE_LUT)
    lab   = np.take(nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32), sidx, axis=SLICE_AXIS)
    lab, _ = _hemisphere_mask(lab, meta["hemisphere"])
    out = os.path.abspath(f"{RUN_DIR}/{subdir}/{tag}_{nm}_slice_{sidx:03d}.svg")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    vol = _bg_volume_for(name_map, nm)                      # <<< EDIT: hoisted out of the call
    if vol is None:
        print(f"    [dump] no underlay for {nm} -> EMPTY svg")   # TROUBLESHOOTING PRINT
    _graph_to_svg(graph, out, t1_volume=vol, slice_index=sidx,   # <<< EDIT: was _bg_volume_for(...)
                  axis=SLICE_AXIS, label_anchors=_label_points_on_slice(lab), table=table,
                  stroke=stroke, line_mode=meta["line_mode"], double_offset=meta["double_offset"],
                  label_detail=meta["label_detail"], hemisphere=meta["hemisphere"],
                  midline_lr=meta["midline_lr"])
    print(f"  [dump] {tag} -> {out}  ({os.path.getsize(out)} bytes)")
    return out

def maybe_dump_after_perturb(corrected, name_map, sidx):
    """One SVG per subject. In mode='anti' the sign is opposite per subject, so a single
    dump shows only half of what the perturbation did."""
    if not DUMP_SVG["after_perturb"]:
        return None
    return [dump_graph_svg(corrected[nm][sidx], sidx, name_map, nm,
                           "after_perturb", "qa_perturbed")
            for nm in sorted(corrected)]

def maybe_dump_after_fuse(sidx):
    if not DUMP_SVG["after_fuse"]:
        return None
    fg = load_state("fused_graphs", default={})
    if sidx not in fg:
        print("  [dump] after_fuse SKIPPED: no fused_graphs state for this slice."); return None
    graph = fg[sidx][0]                       # (BoundaryGraph, seeds)
    # a fused map belongs to no single subject -> no background, which is correct
    return dump_graph_svg(graph, sidx, {"_fused": None}, "_fused", "atlas", "atlas")

def maybe_dump_after_polygonize(sidx):
    if not DUMP_SVG["after_polygonize"]:
        return None
    if not FUSION_PARAMS.build_polygons:
        print("  [dump] after_polygonize SKIPPED: build_polygons=False (no polygons exist).")
        return None
    fr = load_state("fused_regions", default={})
    if not fr.get(sidx):
        print("  [dump] after_polygonize SKIPPED: fused_regions is empty for this slice.")
        return None
    if "render_fused_slice_svg" in globals():               # <<< EDIT: was cell8_render_fused_atlas
        try:
            render_fused_slice_svg()                        # <<< EDIT: same rename
            hits = sorted(glob.glob(f"{RUN_DIR}/atlas/*.svg"))
            print(f"  [dump] after_polygonize -> {os.path.abspath(hits[-1]) if hits else RUN_DIR}")
            return hits[-1] if hits else None
        except Exception as e:
            print(f"  [dump] after_polygonize render error: {e}")
    else:
        print("  [dump] after_polygonize SKIPPED: CELL 8 renderer not defined.")
    return None

# ---- SUPERIMPOSED (background-free) OVERLAY ------------------------------------------
# Every subject's lines in ONE file, one colour and one Inkscape layer each. The
# per-subject dumps cannot show disagreement, because each one is a separate file.
# No raster underlay: the T1 is identical across these subjects, so it only hides lines.
OVERLAY_COLORS = ["#0072b2", "#e69f00", "#009e73", "#cc79a7",
                  "#56b4e9", "#d55e00", "#f0e442", "#999999"]   # Okabe & Ito (2008)


def _canvas_dims():
    """(H_lr, W_si) of one slice; the SVG canvas is width=H_lr, height=W_si, the same frame
    _graph_to_svg uses. Read from the header, so no volume is loaded."""
    shp = nib.load(TEMPLATE_LABELS).shape
    H_lr, W_si = [d for i, d in enumerate(shp) if i != SLICE_AXIS]
    return int(H_lr), int(W_si)


def dump_overlay_svg(layers, path, legend=True):
    """layers: [(label, BoundaryGraph, colour, dash|None, stroke_width), ...] drawn in order."""
    H_lr, W_si = _canvas_dims()
    W, H = float(H_lr), float(W_si)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    out = ['<?xml version="1.0" encoding="UTF-8"?>']
    out.append('<svg xmlns="http://www.w3.org/2000/svg" '
               'xmlns:inkscape="http://www.inkscape.org/namespaces/inkscape" '
               'xmlns:sodipodi="http://sodipodi.sourceforge.net/DTD/sodipodi-0.0.dtd" '
               'viewBox="0 0 %.1f %.1f" width="%.1f" height="%.1f">' % (W, H, W, H))
    out.append('<rect x="0" y="0" width="%.1f" height="%.1f" fill="white"/>' % (W, H))

    for label, g, col, dash, wid in layers:
        sid_ = "".join(ch if ch.isalnum() else "_" for ch in str(label))
        dsh = '' if not dash else ' stroke-dasharray="%s"' % dash
        out.append('<g inkscape:groupmode="layer" inkscape:label="%s" id="layer_%s" '
                   'fill="none" stroke="%s" stroke-width="%.3f" stroke-linejoin="round" '
                   'stroke-linecap="round"%s>' % (sid_, sid_, col, wid, dsh))
        n_arc = 0
        for arc in ef.graph_to_arcs(g):
            P = np.vstack([arc.pts, arc.pts[0]]) if arc.closed else arc.pts
            disp = _orient_arr(P, H_lr, W_si)
            if len(disp) < 2:
                continue
            pts = " ".join("%s,%s" % (_fmt(x), _fmt(y)) for x, y in disp)
            out.append('<polyline points="%s"/>' % pts)
            n_arc += 1
        out.append('</g>')
        print("  [overlay] %-12s %3d arcs  %s" % (label, n_arc, col))   # TROUBLESHOOTING PRINT

    if legend:
        out.append('<g inkscape:groupmode="layer" inkscape:label="legend" id="layer_legend" '
                   'font-family="sans-serif">')
        y = 5.0
        for label, _g, col, dash, wid in layers:
            dsh = '' if not dash else ' stroke-dasharray="%s"' % dash
            out.append('<line x1="3" y1="%.2f" x2="10" y2="%.2f" stroke="%s" '
                       'stroke-width="%.3f"%s/>' % (y, y, col, max(wid, 0.7), dsh))
            out.append('<text x="11.5" y="%.2f" font-size="3.0" fill="black" '
                       'dominant-baseline="central">%s</text>' % (y, _xml_escape(str(label))))
            y += 4.5
        out.append('</g>')

    out.append('</svg>')
    open(path, "w").write("\n".join(out))
    print("  [dump] overlay -> %s  (%d bytes)" % (os.path.abspath(path), os.path.getsize(path)))
    return os.path.abspath(path)


def _overlay_layers(corrected, ren, sidx):
    """Wide grey template underneath (the ground truth), one colour per perturbed subject on
    top. A grey halo rather than a second thin line, so a fused curve drawn last is still
    readable where it coincides with the template."""
    layers = [("template", next(iter(ren.values())), "#b0b0b0", None, 1.20)]
    for j, nm in enumerate(sorted(corrected)):
        layers.append((nm, corrected[nm][sidx], OVERLAY_COLORS[j % len(OVERLAY_COLORS)],
                       None, 0.40))
    return layers


def maybe_dump_perturb_overlay(corrected, ren, sidx):
    """All perturbed subjects superimposed. Companion to the per-subject after_perturb dumps."""
    if not DUMP_SVG.get("perturb_overlay"):
        return None
    return dump_overlay_svg(_overlay_layers(corrected, ren, sidx),
                            "%s/qa_perturbed/overlay_perturbed_slice_%03d.svg" % (RUN_DIR, sidx))


def maybe_dump_fused_overlay(corrected, ren, sidx):
    """The same overlay plus the FUSED lines in black: the fused curve sits in the middle of
    the fan and, in mode='anti', returns onto the grey template."""
    if not DUMP_SVG.get("fused_overlay"):
        return None
    fg = load_state("fused_graphs", default={})
    if sidx not in fg:
        print("  [dump] fused_overlay SKIPPED: no fused_graphs state for this slice.")
        return None
    layers = _overlay_layers(corrected, ren, sidx)
    layers.append(("fused", fg[sidx][0], "#000000", None, 0.50))
    return dump_overlay_svg(layers,
                            "%s/atlas/overlay_fused_slice_%03d.svg" % (RUN_DIR, sidx))


In [ ]:

#@ IMPORTABLE
# =====================================================================================
# THE STAGE RUNNER  +  plan expansion    (machinery; the registry cell only picks values)
# =====================================================================================
FUSION_FIELDS = {f.name for f in _dc.fields(ef.FusionParams)}

# ---- defaults. The REGISTRY cell overrides whatever it wants. -----------------------
N_SEEDS           = 5          # see the Seed-Count Pilot notebook for where 5 comes from
BASE_SEED         = 20260101
SEED_STRIDE       = 100003     # prime, so seed streams cannot alias
SEED_REPEAT_MODES = {"indep"}
EXPORT_QA         = False      # run CELL 6 at the end of the parc stage
IDENTITY_SWEEP    = False      # run ef.sweep_identity (E0.2); costs one fuse per subject
QUIET             = True       # swallow per-run stdout; failures still print
RESUME            = True
FORCE_STRICT_TOPO = True       # project rule: strict_topo is ALWAYS True

BASE_CONTEXT = dict(
    slice_index       = 50,
    hemisphere        = "left",
    smooth_sigma      = 2.0,
    n_subjects        = 2,
    perturb_amp       = 1.0,
    perturb_lambda    = 8.0,
    perturb_mode      = "anti",
    perturb_seed      = BASE_SEED,
    perturb_drop_frac = 0.0,
)

_TOK = {"kp_alpha":"kA", "kp_curv_thresh":"kC", "kp_dmax":"kD", "kp_gamma":"kG",
        "fit_tol_px":"fT", "n_max_seg":"nM", "curv_lambda":"cL", "curv_sigma_px":"cS",
        "curv_sigma_ratio":"cR", "node_tol":"nT", "node_match_max":"nX", "keypoints":"KP",
        "orphan_policy":"OP", "min_arc_support":"mS", "representation":"RP",
        "dangle_repair_px":"dR", "loop_align":"LA", "slice_index":"SL", "hemisphere":"HM",
        "smooth_sigma":"sG", "n_subjects":"N", "perturb_amp":"A", "perturb_lambda":"LM",
        "perturb_mode":"M", "perturb_seed":"sd", "perturb_drop_frac":"DF"}

def _tok(v):
    if isinstance(v, bool): return "T" if v else "F"
    if isinstance(v, float) and v == int(v): return str(int(v))
    return str(v).replace(".", "p").replace("/", "_")

def build_prefix(exp, cfg, swept, seed_idx=None):
    p = "-".join([exp] + [f"{_tok(cfg[k])}{_TOK.get(k, k)}" for k in sorted(swept)])
    return p + (f"-seed{seed_idx}" if seed_idx is not None else "")

def seeds_for(cfg, spec):
    """N_SEEDS replicates whenever the run is SEEDED-STOCHASTIC. That is mode='indep'
    (each subject draws its own field) OR any run that deletes arcs -- drop_arcs is seeded
    too, and at ~120 arcs a single draw is very lumpy (measured: seed 7 deleted the same
    5 of 40 arcs at frac=0.05 and frac=0.15)."""
    n = spec.get("n_seeds", N_SEEDS)
    stochastic = (cfg.get("perturb_mode") in SEED_REPEAT_MODES
                  or cfg.get("perturb_drop_frac", 0.0) > 0
                  or spec.get("repeat_seeds", False))
    if not stochastic or n <= 1:
        return [(int(cfg.get("perturb_seed", BASE_SEED)), None)]
    return [(BASE_SEED + k * SEED_STRIDE, k) for k in range(n)]

def expand(exp, spec):
    """mode='independent' sweeps ONE axis at a time off BASE_CONTEXT.
       mode='combined'    takes the cartesian product (for genuinely coupled parameters)."""
    sweep = {k: v for k, v in spec.get("sweep", {}).items() if v}
    swept = sorted(sweep)
    fixed = spec.get("fixed", {})
    if not sweep:
        combos = [{**BASE_CONTEXT, **fixed}]
    elif spec.get("mode") == "independent":
        combos = [{**BASE_CONTEXT, **fixed, k: v} for k in swept for v in sweep[k]]
    else:
        combos = [{**BASE_CONTEXT, **fixed, **dict(zip(swept, c))}
                  for c in itertools.product(*[sweep[k] for k in swept])]
    out = []
    for c in combos:
        for sd, k in seeds_for(c, spec):
            c2 = dict(c); c2["perturb_seed"] = sd
            out.append((c2, swept, k))
    return out


def apply_config(cfg, stages):
    """Push cfg into TEST's module-level toggles + FUSION_PARAMS.
    (v3 FIX E) build_polygons / run_comparative_diag are DERIVED from `stages`, never
    taken from the sweep, so a plan can no longer ask for S7 while polygonize is off."""
    g = globals()
    for k, G in (("slice_index", "SLICE_INDEX"), ("hemisphere", "HEMISPHERE"),
                 ("smooth_sigma", "SMOOTH_SIGMA"), ("line_mode", "LINE_MODE"),
                 ("trace_max_handles", "TRACE_MAX_HANDLES")):
        if k in cfg: g[G] = cfg[k]
    over = {k: cfg[k] for k in cfg if k in FUSION_FIELDS}

    # (v3 FIX F) curv_sigma_px is DERIVED from kp_alpha by __post_init__ -- but ONLY when
    # it is None. _dc.asdict() of a live FusionParams returns the ALREADY-FILLED value, so
    # sweeping kp_alpha alone silently kept the OLD curvature scale and E4.1 measured
    # alpha against a fixed sigma.
    if cfg.get("curv_sigma_ratio"):
        over["curv_sigma_px"] = cfg["kp_alpha"] / float(cfg["curv_sigma_ratio"])
    elif "kp_alpha" in over and "curv_sigma_px" not in over:
        over["curv_sigma_px"] = None
        print(f"  [params] kp_alpha={over['kp_alpha']} swept -> curv_sigma_px "   # TROUBLESHOOTING PRINT
              f"re-derived as min(1.0, kp_alpha/4)")

    if FORCE_STRICT_TOPO:
        if cfg.get("strict_topo") is False:
            print("  [params] strict_topo=False requested but FORCED True (project rule)")
        over["strict_topo"] = True

    want_poly = "poly" in stages
    over["build_polygons"] = want_poly
    over.setdefault("run_comparative_diag", want_poly)
    g["FUSION_PARAMS"] = ef.FusionParams(**{**_dc.asdict(g["FUSION_PARAMS"]), **over})
    return over


def ensure_lines(cfg, swept, sec):
    """STAGE 'parc'. Full CELL 3/4/5 stack, or restore the frozen post-load cache.
    (v3 FIX G) v2 looked for propagate_to_subjects / cell5_propagate / propagate_labels.
    NONE of those exist -- CELL 5 defines cell5_propagate_single_slice -- so the
    full-stack path silently propagated NOTHING and the next load_state('propagated')
    raised. Only the cache path had ever worked."""
    full, why = needs_full_stack(cfg, swept)
    print(f"  [parc] {'FULL STACK' if full else 'FROM CACHE'} -- {why}")
    if not full:
        with sec("t_cache_load"):
            blob = pickle.load(open(f"{cache_dir_for(cfg)}/postload.pkl", "rb"))
            ren = {k: bg.BoundaryGraph(nodes=n.copy(), elements=[e[:] for e in el])
                   for k, (n, el) in blob["ren"].items()}
            save_state("test_meta", blob["test_meta"])
            save_state("propagated", blob["propagated"])
            if blob.get("regs"):                          # <<< EDIT: CELL 4 is skipped on a
                save_state("regs", blob["regs"])          #     cache hit; dumps need regs
        return ren, blob["name_map"], blob["sidx"]
    with sec("t_parcellation"):
        import_preprocessed()
        cell4_load_registration()
        cell5_propagate_single_slice(simplify_tol=cfg.get("simplify_tol", 0.25))
        if EXPORT_QA:
            cell6_export_single_slice(stroke=0.36)     # "up to where QA export would be"
    ren, name_map, sidx = load_rename_detach()
    d = cache_dir_for(cfg); os.makedirs(d, exist_ok=True)
    pickle.dump({"ren": {k: (np.array(v.nodes), [e[:] for e in v.elements])
                         for k, v in ren.items()},
                 "name_map": name_map, "sidx": sidx,
                 "test_meta": load_state("test_meta"),
                 "propagated": load_state("propagated"),
                 "regs": load_state("regs", default={})},      # <<< EDIT: cache what CELL 4 made
                open(f"{d}/postload.pkl", "wb"))
    print(f"  [cache] wrote {d}/postload.pkl")
    return ren, name_map, sidx


def parc_metrics(ren):
    g0 = next(iter(ren.values()))
    groups = element_arcs(g0)
    codes = {c for c, _, _ in groups}
    return {"P_n_subjects": len(ren), "P_n_nodes": int(len(g0.nodes)),
            "P_n_arcs": len(groups), "P_pts_total": len(g0.elements),
            "P_n_regions": len({r for c in codes for r in c} - {0})}


def run_one(exp, cfg, swept, stages, seed_idx, expect_abort=False):
    """Runs the requested PREFIX of the chain and returns (metrics, timings, aborted)."""
    sec = Sections(); _t0 = time.perf_counter()
    stages = stages_upto(stages)
    prefix = build_prefix(exp, cfg, swept, seed_idx)
    set_run_paths(f"{WORK_TEST}/{_safe_dirname(prefix)}")
    globals()["RUN_PREFIX"] = prefix
    apply_config(cfg, stages)
    M, aborted = {}, False

    # ---------- STAGE 1: parcellation ------------------------------------------------
    ren, name_map, sidx = ensure_lines(cfg, swept, sec)
    M.update(parc_metrics(ren))
    template_arcs = ef.graph_to_arcs(next(iter(ren.values())))    # ground truth for X_tdev
    if "pert" not in stages:
        sec.t["t_total"] = round(time.perf_counter() - _t0, 3)
        return M, sec.t, aborted

    # ---------- STAGE 2: perturbation ------------------------------------------------
    with sec("t_duplicate"):
        ren, name_map = duplicate_to(ren, name_map, cfg.get("n_subjects", len(ren)))
    with sec("t_drop"):
        ren, dstats = drop_arcs(ren, cfg.get("perturb_drop_frac", 0.0), cfg["perturb_seed"])
        M.update(dstats)
    with sec("t_perturb"):
        corrected, _signs, pstats = perturb_all(ren, cfg["perturb_amp"], cfg["perturb_lambda"],
                                                cfg["perturb_mode"], cfg["perturb_seed"], sidx)
        M.update(pstats)
        save_state("corrected", corrected)
    with sec("t_dump_perturb"):
        maybe_dump_after_perturb(corrected, name_map, sidx)
        maybe_dump_perturb_overlay(corrected, ren, sidx)      # <<< EDIT: superimposed dump
    if "fuse" not in stages:
        sec.t["t_total"] = round(time.perf_counter() - _t0, 3)
        return M, sec.t, aborted

    # ---------- STAGE 3/4: fuse (+ polygonize when 'poly' is in stages) --------------
    with sec("t_fuse"):
        try:
            # (v3 FIX H) corrected is passed EXPLICITLY (priority 1). v2 relied on the
            # 'corrected' STATE, and load_state falls back to WORK_TEST/state and
            # WORK/state -- so a run that failed to write its own state silently fused a
            # PREVIOUS run's geometry and reported it as a success.
            cell7_fuse_lines(corrected=corrected, verbose=False)
        except ef.TopologyDivergence as e:
            aborted = True
            M["S5_polygonize_error"] = f"TopologyDivergence: {str(e)[:200]}"
            print(f"  [fuse] ABORTED on topology divergence (expected={expect_abort})")
    if not aborted:
        M.update((load_state("fusion_diag", default={}) or {}).get(sidx, {}))
        res = LAST_FUSION.get("result")
        live = [s for s in (res.stats["arcs"] if res else []) if not s.get("dropped")]
        M.setdefault("S7_budget_saturated", sum(s["saturated"] for s in live))
        M.setdefault("S7_fit_residual_px",
                     round(max((s["fit_residual"] for s in live), default=0.0), 4))
        with sec("t_tdev"):
            M.update(curve_dev(LAST_FUSION.get("arcs") or [], template_arcs))
        if cfg.get("identity_sweep", IDENTITY_SWEEP):
            M.update(ef.sweep_identity({k: v[sidx] for k, v in corrected.items()},
                                       FUSION_PARAMS, step=0.5))
        with sec("t_dump_fuse"):
            maybe_dump_after_fuse(sidx)
            maybe_dump_fused_overlay(corrected, ren, sidx)    # <<< EDIT: overlay + fused line
        with sec("t_dump_polygonize"):
            maybe_dump_after_polygonize(sidx)
    sec.t["t_total"] = round(time.perf_counter() - _t0, 3)
    return M, sec.t, aborted


def execute_plan(plan, jsonl=None, label="plan"):
    """Runs a list of (exp, cfg, swept, stages, seed_idx, prefix, expect_abort) tuples and
    appends one record per run. Shared by the RUN cell and by the Seed-Count Pilot."""
    jsonl = jsonl or RUNS_JSONL
    done = done_prefixes(path=jsonl) if RESUME else set()
    todo = [p for p in plan if p[5] not in done]
    print(f"{label}: {len(plan)} run(s); {len(plan)-len(todo)} already recorded -> "
          f"{len(todo)} to run.\n")
    t0 = time.time()
    for i, (exp, cfg, swept, stages, k, prefix, expect_abort) in enumerate(todo, 1):
        ok, err, aborted, M, T = True, "", False, {}, {}
        try:
            buf = io.StringIO()
            with (contextlib.redirect_stdout(buf) if QUIET else contextlib.nullcontext()):
                M, T, aborted = run_one(exp, cfg, swept, stages, k, expect_abort)
            if aborted and not expect_abort:
                ok, err = False, "TopologyDivergence (strict_topo=True)"
            elif expect_abort and not aborted:
                ok, err = False, "expected an abort and did not get one"
            elif not M:
                ok, err = False, "no metrics produced"
        except Exception as ex:
            ok, err = False, f"{type(ex).__name__}: {ex}"
            traceback.print_exc(limit=3)
        append_run({"experiment": exp, "prefix": prefix, "ok": ok, "err": err,
                    "aborted": aborted, "stages": list(stages), "seed_idx": k,
                    "cfg_hash": _hash({**cfg, "stages": list(stages)}),
                    "swept": {kk: cfg.get(kk) for kk in swept},
                    "config": cfg, "metrics": M, "timings": T}, path=jsonl)
        f = i / len(todo); bar = "#" * int(f * 30) + "." * (30 - int(f * 30))
        print(f"[{bar}] {i}/{len(todo)} {f*100:4.0f}%  {exp:18s} {prefix[:42]:42s} "
              f"{'OK' if ok else 'FAIL ' + err}")
    print(f"\nFinished in {time.time()-t0:.1f}s -> {jsonl}")


def build_plan(active, experiments, manifest_path=None):
    plan, man = [], []
    for e in active:
        spec = experiments[e]
        stages = stages_upto(spec.get("stages", STAGE_ORDER))
        for cfg, swept, k in expand(e, spec):
            pfx = build_prefix(e, cfg, swept, k)
            plan.append((e, cfg, swept, stages, k, pfx, spec.get("expect_abort", False)))
            man.append({"experiment": e, "prefix": pfx, "stages": "+".join(stages),
                        "seed_idx": k, "swept": ";".join(swept),
                        "cfg_hash": _hash({**cfg, "stages": list(stages)}),
                        **{str(kk): vv for kk, vv in sorted(cfg.items())},
                        **{f"fp.{f.name}": getattr(FUSION_PARAMS, f.name)
                           for f in _dc.fields(ef.FusionParams)}})
    if manifest_path:
        import pandas as _pd
        MAN = _pd.DataFrame(man)
        MAN.to_csv(manifest_path, index=False)
        print(f"Manifest: {len(MAN)} run(s) x {len(MAN.columns)} columns -> {manifest_path}")
        dup = MAN[MAN.duplicated("cfg_hash", keep=False)]
        if len(dup):
            print(f"  note: {len(dup)} run(s) share a config with another prefix "
                  f"(all still run; see cfg_hash)")
    return plan


## 3 · ▶ EXPERIMENT REGISTRY — edit this cell

Grouped by experiment number. Each entry is `param -> list of values`; an **empty list means use the default**. `mode="independent"` sweeps one axis at a time off `BASE_CONTEXT`; `mode="combined"` takes the cartesian product (required for genuinely coupled parameters).

**`perturb_lambda` is a wavelength in PIXELS**, not a harmonic count. Set it from the *anatomy* (the smallest feature you refuse to lose), never from `kp_alpha` — otherwise the perturbation changes when the method changes and the keypoints ON/OFF comparison is no longer like-for-like.

In [ ]:

#@ SKIP_ON_IMPORT
# ============================ EDIT ME ============================
# ALPHA_REF / LAMBDA_REF are the anchors this plan is written against. LAMBDA is a
# PHYSICAL property of the perturbation (pixels) and must NOT be derived from kp_alpha at
# run time: keypoints=False has no alpha, and the ON/OFF arms must see the IDENTICAL
# perturbation or the comparison is not like-for-like.
ALPHA_REF  = 4.0     # the kp_alpha you tune around
LAMBDA_REF = 64.0    # px. Perturbation wavelength. >= 6x the ~3.2px vertex spacing (else the
                     # field aliases into per-vertex jitter), and >= 2*pi*amp (else it folds).
DMAX_REF   = 12.0    # the kp_dmax in FUSION_PARAMS; E3.2's A axis is written against it
A_SAFE, A_CRIT, A_BROKEN = 2.0, 7.5, 20.0    # A_BROKEN exceeds LAMBDA_REF/(2*pi)=10.2 -> folds
KP_BEST    = [1.0, 2.0, 4.0]                 # <<< fill in from E4.1 before running E4.4/E4.1b
SLICES     = [50]                           # <<< add your other 2 slices for E0.1

N_SEEDS = 5          # For variance purposes

BASE_CONTEXT = dict(
    slice_index       = SLICES[0],
    hemisphere        = "left",
    smooth_sigma      = 2.0,
    n_subjects        = 2,
    perturb_amp       = 7.5,          # px PEAK per subject; anti/N=2 peak separation = 2A = 15px
    perturb_lambda    = LAMBDA_REF,   # 64px = 9.6mm; 20 samples per vertex spacing
    perturb_mode      = "anti",       # ground-truth case for tuning
    perturb_seed      = BASE_SEED,
    perturb_drop_frac = 0.0,
)

DUMP_SVG["after_perturb"]    = True   # perturbed geometry -> one SVG per subject
DUMP_SVG["perturb_overlay"]  = True   # <<< EDIT: all subjects superimposed, no background
DUMP_SVG["after_fuse"]       = True   # <<< EDIT: fused lines
DUMP_SVG["fused_overlay"]    = True   # <<< EDIT: subjects + fused, superimposed
DUMP_SVG["after_polygonize"] = True   # <<< EDIT: the filled polygon atlas
EXPORT_QA                    = True   # run CELL 6 (QA export) at the end of the parc stage
QUIET                        = False  # let the run's prints through, incl. every dump path

# stages: which prefix of (parc, pert, fuse, poly) to run.
#   'fuse' gives S1-S4 + X_tdev + S7_budget_saturated + S7_fit_residual_px  (cheap)
#   'poly' additionally gives S5, G_*, S6, S7                                (expensive)
EXPERIMENTS = {

  # ONE-OFF VISUAL LOOK (not an experiment). slice_index is swept at a single value so
  # needs_full_stack() forces CELL 3/4/5/6 instead of restoring the parcellation cache.
  "E0_look_perturb": dict(mode="independent", stages=("parc", "pert"),
      sweep=dict(perturb_amp=[7.5]),
      fixed=dict(perturb_lambda=LAMBDA_REF, perturb_mode="anti", n_subjects=2),
      n_seeds=1),

  # ---- E0  IDENTITY CONTROLS  (cheap; run these first, every time) ------------------
  "E0_1_duplicate": dict(mode="independent", stages=("parc","pert","fuse","poly"),
      sweep=dict(slice_index=SLICES),
      fixed=dict(perturb_amp=0.0, n_subjects=2)),
      # two identical copies -> the fused map must BE the input.
      # PASS: X_tdev_p95 <= TDEV_STEP/2, S6_iou_area_weighted ~ 1, G_euler_defect == 0.

  "E0_2_onehot": dict(mode="independent", stages=("parc","pert","fuse"),
      sweep=dict(perturb_amp=[1.0, 2.0]),
      fixed=dict(identity_sweep=True)),
      # ef.sweep_identity: all weight on subject i -> the fused map must return subject i.
      # PASS: S7_sweep_identity_ok == True. Catches resampling / welding / quantisation bugs
      # with no reference data. It exists in TEST v19 CELL 10 and v2 never called it.

  "E0_3_seed_invariance": dict(mode="independent", stages=("parc","pert","fuse"),
      sweep=dict(perturb_amp=[0.0]), repeat_seeds=True, n_seeds=5),
      # at A=0 the field is EXACTLY zero, so all seeds must give BIT-IDENTICAL output.
      # PASS: sd == 0 across the seedX replicates for every metric.

  "E0_4_keypoint_identity": dict(mode="independent", stages=("parc","pert","fuse"),
      sweep=dict(keypoints=[True, False]),
      fixed=dict(perturb_amp=0.0)),
      # identical inputs -> nothing to match -> both arms must produce the same
      # GEOMETRY. VERIFIED in a mock run: the only metrics that differ are
      # S4_kp_anchors_total / S4_kp_anchors_mean, which are bookkeeping (ON has
      # anchors, OFF has none). PASS = every OTHER metric identical, in particular
      # X_tdev_*, S1-S3 and S4_snap_dist_max_px. If any of those differ, the
      # key-point machinery is moving geometry on its own.

  # E0.5  FUSION VISUAL DEMO. 6 subjects, anti mode, defaults except the amplitude sweep.
  # Every subject is an identical copy of the template before perturbation (duplicate_to),
  # so the fan in the overlay is the perturbation and nothing else. signs sum to zero, so
  # the fused curve has a known ground truth: the template.
  "E0_5_fusion_demo": dict(mode="independent", stages=("parc","pert","fuse","poly"),
      sweep=dict(perturb_amp=[A_SAFE, A_CRIT]),
      fixed=dict(n_subjects=6, perturb_mode="anti", perturb_lambda=LAMBDA_REF),
      n_seeds=1),

  # ---- E3  PERTURBATION: where does the pipeline break? ----------------------------
  "E3_2_screen": dict(mode="combined", stages=("parc","pert","fuse"),      # 7 x 2 = 14
      sweep=dict(perturb_amp=[2.0, 4.0, 7.5, 10.0, DMAX_REF, 2*DMAX_REF, 3*DMAX_REF],
                 keypoints=[True, False]),
      fixed=dict(perturb_mode="anti", perturb_lambda=LAMBDA_REF)),

  "E3_3_grid": dict(mode="combined", stages=("parc","pert","fuse","poly"), # 3x3x2x2 = 36
      sweep=dict(perturb_amp=[A_SAFE, A_CRIT, A_BROKEN],
                 perturb_lambda=[LAMBDA_REF/2, LAMBDA_REF, 2*LAMBDA_REF],
                 perturb_mode=["anti", "indep"],
                 keypoints=[True, False])),
      # indep arms get N_SEEDS replicates each -> 24 anti + 24*N_SEEDS indep runs.

  "E3_4_same": dict(mode="combined", stages=("parc","pert","fuse","poly"),
      sweep=dict(perturb_amp=[A_SAFE, A_CRIT, A_BROKEN]),
      fixed=dict(perturb_mode="same", perturb_lambda=LAMBDA_REF)),
      # all subjects displaced identically -> the fused map must equal the displaced input,
      # i.e. a translated identity control. X_tdev here measures the DISPLACEMENT, not error.

  "E3_5_node_match_max": dict(mode="independent", stages=("parc","pert","fuse"),
      sweep=dict(node_match_max=[5.0, 10.0, 20.0, 35.0, 50.0, 100.0]),
      fixed=dict(perturb_amp=A_CRIT)),
      # every metric here is S3 -> no polygons needed. Compare against
      # S3_suggested_node_match_max (= 3 x S3_node_disp_p95), which the run prints for you.

  # ---- E4  PARAMETER DETERMINATION -------------------------------------------------
  "E4_1_kp_alpha": dict(mode="combined", stages=("parc","pert","fuse"),
      sweep=dict(kp_alpha=[0.5, 1.0, 2.0, 4.0, 8.0, 12.0], perturb_amp=[4.0, 7.5])),
      # 0.5 ADDED: your CELL A notes measure alpha=2 beating 4 beating 8, so the minimum
      # is not yet bracketed. curv_sigma_px is now re-derived per alpha (v3 FIX F).

  "E4_1b_confirm": dict(mode="combined", stages=("parc","pert","fuse","poly"),
      sweep=dict(kp_alpha=KP_BEST, perturb_amp=[4.0, 7.5])),
      # the 3 best from E4_1, re-run WITH polygons for the S6/S7 shape verdict.

  "E4_2_fit_tol": dict(mode="combined", stages=("parc","pert","fuse"),
      sweep=dict(fit_tol_px=[0.05, 0.1, 0.2, 0.5, 1.0, 2.0], perturb_amp=[4.0, 7.5])),
      # S7_fit_residual_px + S4_pts_total, both available at the fuse stage.
      # NOTE flatten_px must stay <= fit_tol_px or __post_init__ warns; it is INERT here
      # anyway (it is only consumed by read_qa_svg, which the runner never calls).

  "E4_3_nmax_seg": dict(mode="independent", stages=("parc","pert","fuse"),
      sweep=dict(n_max_seg=[200, 100, 80, 60, 40, 20, 10])),
      # find the floor via S7_budget_saturated (how many arcs hit the clamp).

  "E4_4_alpha_sigma": dict(mode="combined", stages=("parc","pert","fuse"),
      sweep=dict(kp_alpha=KP_BEST, curv_sigma_ratio=[2, 4, 8])),
      # curv_sigma_px = kp_alpha / ratio, computed per run (this is what v2's
      # curv_sigma_px=[None] placeholder was meant to do and never did).
      # ratio=2 will WARN from __post_init__ -- that warning IS the boundary being tested.

  "E4_5_kp_gamma": dict(mode="independent", stages=("parc","pert","fuse","poly"),
      sweep=dict(kp_gamma=[0.0, 1/3, 2/3, 1.0]),
      fixed=dict(perturb_mode="indep")),

  "E4_6_curv_thresh": dict(mode="independent", stages=("parc","pert","fuse"),
      sweep=dict(kp_curv_thresh=[0.0, 0.025, 0.05, 0.1, 0.2, 0.5, 1.0])),
      # STOPS AT 1.0 deliberately. Curvature is 1/px and curv_sigma_px ~ 1, so a real
      # notch is |kappa| ~ 0.5-1. Thresholds of 2/4/8 kill essentially every key point,
      # which is exactly the keypoints=False arm you already run.

  "E4_7_kp_dmax": dict(mode="combined", stages=("parc","pert","fuse"),
      sweep=dict(kp_dmax=[3.0, 6.0, 9.0, 12.0, 18.0, 24.0], perturb_amp=[4.0, 7.5],
                 perturb_mode=["anti", "indep"]),),
      # E4.7.1: if S4_kp_anchors_mean is still climbing at 8, extend the list and re-run;
      # the resume registry will only execute the new values.

  "E4_8_dangle": dict(mode="combined", stages=("parc","pert","fuse","poly"),
      sweep=dict(dangle_repair_px=[0.0, 1.5], perturb_drop_frac=[0.15])),
      # drop_frac > 0 is MANDATORY here. Dangling ends come from QA gaps or deleted arcs;
      # the synthetic path only MOVES nodes, so with drop_frac=0 there are no dangles and
      # both values return byte-identical output. That is why E4.8 showed nothing in v2.

  "E4_9_curv_lambda": dict(mode="independent", stages=("parc","pert","fuse"),
      sweep=dict(curv_lambda=[0.0, 0.5, 1.0])),

  "E4_10_representation": dict(mode="combined", stages=("parc","pert","fuse","poly"),
      sweep=dict(representation=["polyline", "spline"], perturb_amp=[1.0, 2.0])),
      # spline_smooth stays 0. splprep's s is NOT scale-free -- it depends on point count
      # and residual magnitude, so one swept value does not mean the same thing on two
      # different arcs and the numbers would not be comparable.

  "E4_11_node_tol": dict(mode="independent", stages=("parc","pert","fuse","poly"),
      sweep=dict(node_tol=[0.5, 1.0, 2.0, 3.0])),
      # ADDED: nothing in the plan touched it, yet it sets all within-subject junction
      # welding and half of the S7_sweep_identity bound (fit_tol*3 + node_tol/2).

  # ---- E5  SCALING -----------------------------------------------------------------
  "E5_1_scaling": dict(mode="combined", stages=("parc","pert","fuse","poly"), n_seeds=3,
      sweep=dict(n_subjects=[2, 3, 4],
                 perturb_amp=[0.0, 0.25, 0.5, 1.0, 2.0, 5.0, 8.0]),
      fixed=dict(perturb_mode="indep")),
      # 21 cells x 3 seeds = 63 runs WITH polygons -- the most expensive block in the plan.
      # n_seeds is dropped to 3 here on purpose; the CI will be ~2x wider than at 5.
      # Add n_subjects=5 if runtime allows: SATM claims the measures stabilise at n ~ 3,
      # and N=5 is what DEMONSTRATES the plateau instead of assuming it.

  # ---- E6  ORPHAN ARCS -------------------------------------------------------------
  "E6_1_orphans": dict(mode="combined", stages=("parc","pert","fuse","poly"),
      sweep=dict(perturb_drop_frac=[0.05, 0.15, 0.25], perturb_amp=[1.0, 2.0],
                 orphan_policy=["passthrough", "drop", "reference"])),
      # orphan_policy and min_arc_support ADDED: drop_frac is the ENABLER, not the thing
      # under test. Every run here is seed-replicated because drop_arcs is seeded.
      # PASS/FAIL lives in S5_unlabeled_faces, S5_orphan_arcs, S5_region_yield.
}

ACTIVE = ["E0_5_fusion_demo"]

print(f"Registry loaded. ACTIVE={ACTIVE}")
print(f"  N_SEEDS={N_SEEDS}  ALPHA_REF={ALPHA_REF}  LAMBDA_REF={LAMBDA_REF}  "
      f"strict_topo FORCED {FORCE_STRICT_TOPO}")
print(f"  available: {list(EXPERIMENTS)}")


## 4 · ▶ RUN

Builds the plan, writes the full-detail manifest, then per run: applies parameters, enters the pipeline at the right point, runs forward to the requested stage, and appends one record to `runs.jsonl`. Re-running skips anything already recorded.

In [ ]:

#@ SKIP_ON_IMPORT
# =====================================================================================
# RUN
# =====================================================================================
PLAN = build_plan(ACTIVE, EXPERIMENTS, manifest_path=f"{RESULTS_DIR}/run_manifest.csv")
execute_plan(PLAN, label="Plan")


## 5 · REPORT

Rebuilds every derived view from `runs.jsonl`: wide CSV, tidy CSV, seed-CI summary, a multi-sheet `metrics.xlsx`, and CI plots. Safe to run at any time, including mid-batch.

In [ ]:

#@ IMPORTABLE
# =====================================================================================
# REPORT MACHINERY  (everything here is DERIVED from runs.jsonl; safe to delete/rebuild)
# =====================================================================================
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats as _st
from openpyxl.styles import PatternFill                    # <<< EDIT: workbook via openpyxl
from openpyxl.utils import get_column_letter               #     (xlsxwriter is not installed)
from openpyxl.formatting.rule import CellIsRule

_SCALARS = (int, float, bool, str, type(None))
_IDCOLS  = ("experiment", "prefix", "ok", "err", "aborted", "stages", "seed_idx", "cfg_hash")
HARD     = ["G_ed", "S5_mr", "S7_td", "S5_uf", "S5_oa", "S5_du", "S5_ac", "S3_nrf", "S4_ad"]

def _flatten(rec):
    out = {c: rec.get(c) for c in _IDCOLS}
    out["stages"] = "+".join(rec.get("stages", []))
    for k, v in (rec.get("swept") or {}).items():
        out[f"p.{k}"] = v
    for src in ("metrics", "timings"):
        for k, v in (rec.get(src) or {}).items():
            out[METRIC_KEY.get(k, k)] = v if isinstance(v, _SCALARS) else json.dumps(v, default=str)
    return out

def build_wide(path=None, out_dir=None):
    out_dir = out_dir or RESULTS_DIR
    recs = read_runs(path)
    if not recs:
        print("runs.jsonl is empty."); return pd.DataFrame()
    W = pd.DataFrame([_flatten(r) for r in recs]).drop_duplicates("prefix", keep="last")
    W.to_csv(f"{out_dir}/metrics_wide.csv", index=False)
    return W

def build_long(W, out_dir=None):
    out_dir = out_dir or RESULTS_DIR
    idc = [c for c in W.columns if c in _IDCOLS or c.startswith("p.")]
    L = W.melt(id_vars=idc, var_name="metric", value_name="value").dropna(subset=["value"])
    L.to_csv(f"{out_dir}/metrics_long.csv", index=False)
    return L

def build_summary(W, conf=0.95, out_dir=None):
    """Mean +/- Student-t CI across the seedX replicates of ONE configuration."""
    out_dir = out_dir or RESULTS_DIR
    num = [c for c in W.columns if c not in _IDCOLS and not c.startswith("p.")
           and pd.api.types.is_numeric_dtype(W[c])]
    grp = ["experiment", "stages"] + [c for c in W.columns if c.startswith("p.")]
    rows = []
    for keys, sub in W[W.ok == True].groupby(grp, dropna=False):
        base = dict(zip(grp, keys if isinstance(keys, tuple) else (keys,)))
        for m in num:
            v = pd.to_numeric(sub[m], errors="coerce").dropna().to_numpy(float)
            if v.size == 0: continue
            n, mu = v.size, float(v.mean())
            if n > 1:
                sd = float(v.std(ddof=1))
                hw = float(_st.t.ppf(0.5 + conf/2, n-1) * sd / math.sqrt(n))
            else:
                sd, hw = 0.0, float("nan")
            rows.append({**base, "metric": m, "n_seeds": n, "mean": round(mu, 6),
                         "sd": round(sd, 6),
                         "ci_lo": round(mu-hw, 6) if n > 1 else None,
                         "ci_hi": round(mu+hw, 6) if n > 1 else None,
                         "min": round(float(v.min()), 6), "max": round(float(v.max()), 6)})
    S = pd.DataFrame(rows)
    S.to_csv(f"{out_dir}/metrics_summary.csv", index=False)
    return S

def build_excel(W, S, out_dir=None):
    """One workbook: KEY | SUMMARY | one sheet per experiment. Filters, frozen headers,
    and every HARD-threshold column shaded red where non-zero. Rebuilt from scratch every
    time, so it can never drift from runs.jsonl."""
    out_dir = out_dir or RESULTS_DIR
    key = pd.read_csv(METRIC_KEY_CSV)
    path = f"{out_dir}/metrics.xlsx"
    red = PatternFill(start_color="FFFFC7CE", end_color="FFFFC7CE", fill_type="solid")
    with pd.ExcelWriter(path, engine="openpyxl") as xw:    # <<< EDIT: was engine="xlsxwriter"
        key.to_excel(xw, sheet_name="KEY", index=False)
        S.to_excel(xw, sheet_name="SUMMARY", index=False)
        for exp, sub in W.groupby("experiment"):
            sh = str(exp)[:31]
            sub.to_excel(xw, sheet_name=sh, index=False)
            ws = xw.sheets[sh]
            ws.freeze_panes = "C2"                          # header row + experiment/prefix
            last_col = get_column_letter(max(len(sub.columns), 1))
            ws.auto_filter.ref = f"A1:{last_col}{len(sub) + 1}"
            for j, c in enumerate(sub.columns, start=1):
                if c in HARD and len(sub):
                    col = get_column_letter(j)
                    ws.conditional_formatting.add(
                        f"{col}2:{col}{len(sub) + 1}",
                        CellIsRule(operator="notEqual", formula=["0"], fill=red))
    print(f"  -> {path}")
    return path

def build_plots(S, metrics=("X_tdp", "S7_rdm", "S7_pdms", "S6_iaw", "T_tot"), out_dir=None):
    out_dir = out_dir or RESULTS_DIR
    made = []
    for exp, sub in S.groupby("experiment"):
        pcols = [c for c in sub.columns if c.startswith("p.") and sub[c].nunique() > 1]
        for pc in pcols:
            for m in metrics:
                d = sub[sub.metric == m].dropna(subset=[pc])
                if len(d) < 2: continue
                d = d.sort_values(pc)
                fig, ax = plt.subplots(figsize=(5, 3.2))
                yerr = (d["mean"] - d["ci_lo"]).fillna(0.0).to_numpy(float)
                ax.errorbar(d[pc].astype(str), d["mean"], yerr=yerr, marker="o", capsize=3)
                ax.set_xlabel(pc[2:]); ax.set_ylabel(m)
                ax.set_title(f"{exp}: {m} vs {pc[2:]}", fontsize=9)
                ax.grid(alpha=.3); fig.tight_layout()
                p = f"{out_dir}/plots/{exp}__{pc[2:]}__{m}.png"
                fig.savefig(p, dpi=110); plt.close(fig); made.append(p)
    print(f"  -> {len(made)} plot(s) in {out_dir}/plots/")
    return made

def rebuild_report(path=None, out_dir=None):
    W = build_wide(path, out_dir)
    if W.empty: return None
    L = build_long(W, out_dir); S = build_summary(W, out_dir=out_dir)
    try:
        build_excel(W, S, out_dir)
    except Exception as e:                                  # <<< EDIT: workbook is a derived
        print(f"  [report] metrics.xlsx SKIPPED "           #     view; CSVs must still land
              f"({type(e).__name__}: {e}); the CSVs are unaffected.")
    build_plots(S, out_dir=out_dir)
    cols = [c for c in HARD if c in W.columns]
    n_bad = int((W[cols].apply(pd.to_numeric, errors="coerce").fillna(0) != 0)
                .any(axis=1).sum()) if cols else 0
    print(f"\n{len(W)} run(s) | {len(S)} summarised config-metric pairs | "
          f"{n_bad} run(s) violate a HARD threshold | "
          f"{int((~W.ok.astype(bool)).sum())} failed")
    return W, L, S


In [ ]:

#@ SKIP_ON_IMPORT
DF, LONG, SUMMARY = rebuild_report()
DF.head(30)
